In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

In [2]:
PROJECT_ROOT = Path("..")

RAW_LEXICON_FILE = (
    PROJECT_ROOT
    / "data"
    / "expansion"
    / "self_assertion_expanded_raw_primary.csv"
)

DEV_SENTENCES_FILE = (
    PROJECT_ROOT
    / "data"
    / "preprocessed"
    / "development_sentences_clean.csv"
)

PURIFICATION_DIR = (
    PROJECT_ROOT
    / "data"
    / "purification"
)

PURIFICATION_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [3]:
raw_lexicon = pd.read_csv(RAW_LEXICON_FILE)
dev_sentences = pd.read_csv(DEV_SENTENCES_FILE)

print("Raw lexicon terms:", len(raw_lexicon))
print("Development sentences:", len(dev_sentences))

Raw lexicon terms: 63791
Development sentences: 157346


In [4]:
print(dev_sentences.columns.tolist())

['corpus', 'iso3', 'session', 'year', 'filename', 'sentence_id', 'sentence_text', 'sentence_text_lower', 'procedural_intro']


In [5]:
dev_sentences[
    [
        "iso3",
        "year",
        "filename",
        "sentence_id",
        "sentence_text_lower"
    ]
].head()

,iso3,year,filename,sentence_id,sentence_text_lower
0,AFG,2016,AFG_71_2016.txt,0,it is my distinct pleasure to attend the seven...
1,AFG,2016,AFG_71_2016.txt,1,i sincerely congratulate ambassador peter thom...
2,AFG,2016,AFG_71_2016.txt,2,i thank secretary-general ban ki-moon for his ...
3,AFG,2016,AFG_71_2016.txt,3,we wish him every success in his future endeav...
4,AFG,2016,AFG_71_2016.txt,4,the twenty-first century has brought with it a...


In [6]:
import re
from collections import Counter, defaultdict

In [7]:
TOKEN_PATTERN = re.compile(r"[a-z]+(?:-[a-z]+)*")

def tokenize_for_kgl(text):
    return TOKEN_PATTERN.findall(str(text).lower())

In [8]:
test_text = "We reaffirm our position, reject the proposal, and defend self-assertion."

print(tokenize_for_kgl(test_text))

['we', 'reaffirm', 'our', 'position', 'reject', 'the', 'proposal', 'and', 'defend', 'self-assertion']


In [9]:
lexicon_terms = set(
    raw_lexicon["term"]
    .astype(str)
    .str.lower()
)

print("Raw lexicon lookup size:", len(lexicon_terms))

Raw lexicon lookup size: 63161


In [10]:
raw_lexicon["term_lower"] = (
    raw_lexicon["term"]
    .astype(str)
    .str.lower()
)

case_duplicate_counts = (
    raw_lexicon
    .groupby("term_lower")
    .size()
    .sort_values(ascending=False)
)

print(
    "Lowercase terms represented by multiple raw entries:",
    (case_duplicate_counts > 1).sum()
)

case_duplicate_counts[
    case_duplicate_counts > 1
].head(30)

Lowercase terms represented by multiple raw entries: 627


term_lower
a-okay           3
pov-check        3
jewwatch         3
re-check         2
unprecedented    2
purists          2
cooperate        2
mantion          2
castigate        2
commensurate     2
prepend          2
diminish         2
purporting       2
commends         2
castrate         2
purposefully     2
ascribe          2
nullify          2
ascertain        2
smugly           2
manipulate       2
whate            2
unproven         2
reiterate        2
manifestly       2
reinvigorate     2
convoke          2
strenuously      2
westernize       2
novacatz         2
dtype: int64

In [11]:
duplicate_lower_terms = set(
    case_duplicate_counts[
        case_duplicate_counts > 1
    ].index
)

raw_lexicon[
    raw_lexicon["term_lower"].isin(
        list(duplicate_lower_terms)[:20]
    )
].sort_values(
    ["term_lower", "term"]
)

,term,weight,sources,term_lower
48592,Admonished,0.468046,concept,admonished
20376,admonished,0.518460,concept,admonished
37776,Colonize,0.483436,concept,colonize
31415,colonize,0.494600,concept,colonize
63248,Complementarian,0.450612,concept,complementarian
7167,complementarian,0.568359,concept,complementarian
36097,Comprehend,0.486226,concept,comprehend
8161,comprehend,0.562929,concept,comprehend
32161,Criminalise,0.493266,concept,criminalise
14022,criminalise,0.537262,concept,criminalise


In [12]:
normalized_lexicon = (
    raw_lexicon
    .assign(
        term=raw_lexicon["term"]
        .astype(str)
        .str.lower()
    )
    .groupby(
        "term",
        as_index=False
    )
    .agg(
        weight=("weight", "max"),
        sources=(
            "sources",
            lambda x: "; ".join(
                sorted(
                    set(
                        source
                        for value in x.dropna()
                        for source in str(value).split("; ")
                    )
                )
            )
        )
    )
)

In [13]:
print(
    "Raw expanded terms before lowercase normalization:",
    len(raw_lexicon)
)

print(
    "Unique terms after lowercase normalization:",
    len(normalized_lexicon)
)

Raw expanded terms before lowercase normalization: 63791
Unique terms after lowercase normalization: 63161


In [14]:
seed_terms = {
    "self-assertion",
    "assert",
    "urge",
    "propose",
    "advocate",
    "declare",
    "defend",
    "reaffirm",
    "insist",
    "reject",
    "resist",
    "intend",
    "ready",
    "exercise"
}

missing_seeds = (
    seed_terms
    - set(normalized_lexicon["term"])
)

print("Missing seeds:", missing_seeds)

Missing seeds: set()


In [15]:
normalized_lexicon[
    normalized_lexicon["term"].isin(seed_terms)
].sort_values("term")

,term,weight,sources
1118,advocate,1.0,concept; seed
3386,assert,1.0,concept; seed
13226,declare,1.0,concept; seed
13445,defend,1.0,concept; seed
18850,exercise,1.0,concept; seed
27032,insist,1.0,concept; seed
27282,intend,1.0,concept; seed
41981,propose,1.0,concept; seed
44670,ready,1.0,concept; seed
44684,reaffirm,1.0,concept; seed


In [16]:
lexicon_terms = set(
    normalized_lexicon["term"]
)

print(
    "Normalized lexicon lookup size:",
    len(lexicon_terms)
)

Normalized lexicon lookup size: 63161


In [17]:
import re
from collections import Counter, defaultdict

In [18]:
TOKEN_PATTERN = re.compile(r"[a-z]+(?:-[a-z]+)*")

def tokenize_for_kgl(text):
    return TOKEN_PATTERN.findall(str(text).lower())

In [19]:
test_text = "We reaffirm our position, reject the proposal, and defend self-assertion."

print(tokenize_for_kgl(test_text))

['we', 'reaffirm', 'our', 'position', 'reject', 'the', 'proposal', 'and', 'defend', 'self-assertion']


In [20]:
token_counts = Counter()

sentence_sets = defaultdict(set)
speech_sets = defaultdict(set)
country_sets = defaultdict(set)
country_year_sets = defaultdict(set)

for row in dev_sentences.itertuples(index=False):

    tokens = tokenize_for_kgl(row.sentence_text_lower)

    matched_tokens = [
        token
        for token in tokens
        if token in lexicon_terms
    ]

    if not matched_tokens:
        continue

    token_counts.update(matched_tokens)

    unique_terms = set(matched_tokens)

    sentence_key = (row.filename, row.sentence_id)
    speech_key = row.filename
    country_key = row.iso3
    country_year_key = (row.iso3, row.year)

    for term in unique_terms:
        sentence_sets[term].add(sentence_key)
        speech_sets[term].add(speech_key)
        country_sets[term].add(country_key)
        country_year_sets[term].add(country_year_key)

In [21]:
frequency_rows = []

for term in normalized_lexicon["term"]:

    frequency_rows.append({
        "term": term,
        "token_count": token_counts.get(term, 0),
        "sentence_count": len(sentence_sets.get(term, set())),
        "speech_count": len(speech_sets.get(term, set())),
        "country_count": len(country_sets.get(term, set())),
        "country_year_count": len(
            country_year_sets.get(term, set())
        )
    })

frequency_df = pd.DataFrame(frequency_rows)

In [22]:
frequency_df = frequency_df.merge(
    normalized_lexicon,
    on="term",
    how="left"
)

In [23]:
print("Raw normalized terms:", len(frequency_df))

for cutoff in [1, 2, 5, 10, 20, 50, 100]:
    print(
        f"Terms occurring at least {cutoff} times:",
        (frequency_df["token_count"] >= cutoff).sum()
    )

Raw normalized terms: 63161
Terms occurring at least 1 times: 7918
Terms occurring at least 2 times: 6310
Terms occurring at least 5 times: 4864
Terms occurring at least 10 times: 3993
Terms occurring at least 20 times: 3158
Terms occurring at least 50 times: 2184
Terms occurring at least 100 times: 1601


In [24]:
frequency_df["token_count"].describe(
    percentiles=[.50, .75, .90, .95, .99]
)

count     63161.000000
mean         36.717531
std        1462.844056
min           0.000000
50%           0.000000
75%           0.000000
90%           1.000000
95%          19.000000
99%         409.400000
max      266365.000000
Name: token_count, dtype: float64

In [25]:
check_terms = [
    "assert",
    "urge",
    "ready",
    "accept",
    "implore",
    "self-assertion",
    "exercise",
    "defend"
]

frequency_df[
    frequency_df["term"].isin(check_terms)
][
    [
        "term",
        "token_count",
        "sentence_count",
        "speech_count",
        "country_count",
        "country_year_count",
        "weight",
        "sources"
    ]
].sort_values(
    "token_count",
    ascending=False
)

,term,token_count,sentence_count,speech_count,country_count,country_year_count,weight,sources
60394,urge,697,694,488,155,488,1.000000,concept; seed
44670,ready,666,659,483,152,483,1.000000,concept; seed
13445,defend,440,433,299,126,299,1.000000,concept; seed
383,accept,347,345,258,125,258,0.800865,concept; synonym:reject
18850,exercise,310,309,255,118,255,1.000000,concept; seed
3386,assert,37,37,33,30,33,1.000000,concept; seed
25644,implore,23,23,22,19,22,0.773136,concept; synonym:urge
49729,self-assertion,1,1,1,1,1,1.000000,concept; seed


In [26]:
cutoff_diagnostics = []

for cutoff in [1, 2, 5, 10, 20, 50, 100]:
    
    subset = frequency_df[
        frequency_df["token_count"] >= cutoff
    ]
    
    cutoff_diagnostics.append({
        "token_cutoff": cutoff,
        "n_terms": len(subset),
        "median_token_count": subset["token_count"].median(),
        "median_country_count": subset["country_count"].median(),
        "mean_country_count": subset["country_count"].mean(),
        "n_terms_5plus_countries": (
            subset["country_count"] >= 5
        ).sum(),
        "n_terms_10plus_countries": (
            subset["country_count"] >= 10
        ).sum(),
        "n_terms_20plus_countries": (
            subset["country_count"] >= 20
        ).sum(),
        "n_terms_30plus_countries": (
            subset["country_count"] >= 30
        ).sum()
    })

cutoff_diagnostics = pd.DataFrame(cutoff_diagnostics)

cutoff_diagnostics

,token_cutoff,n_terms,median_token_count,median_country_count,mean_country_count,n_terms_5plus_countries,n_terms_10plus_countries,n_terms_20plus_countries,n_terms_30plus_countries
0,1,7918,10.0,8.0,34.187926,4728,3835,2930,2432
1,2,6310,20.0,16.0,42.645325,4728,3835,2930,2432
2,5,4864,39.0,29.5,54.559827,4728,3835,2930,2432
3,10,3993,62.0,44.0,65.164538,3983,3835,2930,2432
4,20,3158,102.0,63.0,79.185244,3155,3150,2930,2432
5,50,2184,189.5,98.0,103.299451,2184,2183,2177,2171
6,100,1601,297.0,120.0,123.126796,1601,1600,1599,1598


In [27]:
frequency_df.loc[
    frequency_df["token_count"] > 0,
    [
        "term",
        "token_count",
        "country_count",
        "speech_count",
        "weight",
        "sources"
    ]
].sort_values(
    ["token_count", "country_count"],
    ascending=[True, True]
).head(50)

,term,token_count,country_count,speech_count,weight,sources
165,abates,1,1,1,0.487986,concept
196,abidance,1,1,1,0.521517,concept
235,abnegation,1,1,1,0.507911,concept
278,abrogates,1,1,1,0.505690,concept
279,abrogating,1,1,1,0.478053,concept
307,absolved,1,1,1,0.459113,concept
308,absolves,1,1,1,0.505119,concept
460,accomplis,1,1,1,0.483262,concept
503,accredit,1,1,1,0.458849,concept
640,acquiescence,1,1,1,0.541538,concept


In [28]:
frequency_df.loc[
    frequency_df["token_count"] > 0,
    "country_count"
].describe(
    percentiles=[.25, .50, .75, .90, .95, .99]
)

count    7918.000000
mean       34.187926
std        49.927443
min         1.000000
25%         2.000000
50%         8.000000
75%        45.000000
90%       121.000000
95%       161.150000
99%       178.000000
max       178.000000
Name: country_count, dtype: float64

In [29]:
seed_diagnostics = frequency_df[
    frequency_df["sources"].str.contains(
        "seed",
        na=False
    )
][
    [
        "term",
        "token_count",
        "sentence_count",
        "speech_count",
        "country_count",
        "country_year_count",
        "weight",
        "sources"
    ]
].sort_values(
    "token_count",
    ascending=False
)

seed_diagnostics

,term,token_count,sentence_count,speech_count,country_count,country_year_count,weight,sources
60394,urge,697,694,488,155,488,1.0,concept; seed
44670,ready,666,659,483,152,483,1.0,concept; seed
44684,reaffirm,648,647,455,147,455,1.0,concept; seed
13445,defend,440,433,299,126,299,1.0,concept; seed
18850,exercise,310,309,255,118,255,1.0,concept; seed
1118,advocate,306,304,255,119,255,1.0,concept; seed
46253,reject,267,261,183,87,183,1.0,concept; seed
41981,propose,142,141,110,64,110,1.0,concept; seed
27282,intend,136,134,105,70,105,1.0,concept; seed
13226,declare,82,82,76,62,76,1.0,concept; seed


In [30]:
print(cutoff_diagnostics.to_string(index=False))

 token_cutoff  n_terms  median_token_count  median_country_count  mean_country_count  n_terms_5plus_countries  n_terms_10plus_countries  n_terms_20plus_countries  n_terms_30plus_countries
            1     7918                10.0                   8.0           34.187926                     4728                      3835                      2930                      2432
            2     6310                20.0                  16.0           42.645325                     4728                      3835                      2930                      2432
            5     4864                39.0                  29.5           54.559827                     4728                      3835                      2930                      2432
           10     3993                62.0                  44.0           65.164538                     3983                      3835                      2930                      2432
           20     3158               102.0                  

In [31]:
cutoff_diagnostics.to_csv(
    PURIFICATION_DIR / "self_assertion_cutoff_diagnostics.csv",
    index=False
)

seed_diagnostics.to_csv(
    PURIFICATION_DIR / "self_assertion_seed_diagnostics.csv",
    index=False
)

frequency_df.to_csv(
    PURIFICATION_DIR / "self_assertion_frequency_diagnostics.csv",
    index=False
)

In [32]:
FREQUENCY_THRESHOLD = 10

In [34]:
SEED_FILE = (
    PROJECT_ROOT
    / "data"
    / "seed_candidates"
    / "self_assertion_seed_set_v1.csv"
)

seed_df = pd.read_csv(SEED_FILE)

print(seed_df.head())
print("Number of seeds:", len(seed_df))

       seed              facet    source final_decision
0      urge    advance_outcome  original           keep
1   propose    advance_outcome  original           keep
2    reject  resist_constraint  original           keep
3  advocate    advance_outcome  original           keep
4  reaffirm  maintain_position  original           keep
Number of seeds: 14


In [35]:
seed_terms = set(
    seed_df["seed"]
    .astype(str)
    .str.lower()
)

In [36]:
print(seed_terms)
print("Number of seed terms:", len(seed_terms))

{'defend', 'reject', 'exercise', 'propose', 'self-assertion', 'declare', 'advocate', 'assert', 'intend', 'insist', 'resist', 'ready', 'reaffirm', 'urge'}
Number of seed terms: 14


In [37]:
frequency_df["is_seed"] = (
    frequency_df["term"].isin(seed_terms)
)

frequency_df["passes_frequency"] = (
    frequency_df["token_count"] >= FREQUENCY_THRESHOLD
)

In [38]:
frequency_df["retain_after_frequency"] = (
    frequency_df["passes_frequency"]
    | frequency_df["is_seed"]
)

In [39]:
frequency_purified = (
    frequency_df[
        frequency_df["retain_after_frequency"]
    ]
    .copy()
    .reset_index(drop=True)
)

In [40]:
print(
    "Terms before frequency purification:",
    len(frequency_df)
)

print(
    "Terms after frequency purification:",
    len(frequency_purified)
)

print(
    "Expanded terms meeting frequency threshold:",
    (
        frequency_df["passes_frequency"]
        & ~frequency_df["is_seed"]
    ).sum()
)

print(
    "Seeds protected despite falling below threshold:",
    (
        frequency_df["is_seed"]
        & ~frequency_df["passes_frequency"]
    ).sum()
)

Terms before frequency purification: 63161
Terms after frequency purification: 3994
Expanded terms meeting frequency threshold: 3980
Seeds protected despite falling below threshold: 1


In [41]:
frequency_purified[
    frequency_purified["is_seed"]
][
    [
        "term",
        "token_count",
        "country_count",
        "weight",
        "passes_frequency"
    ]
].sort_values(
    "token_count",
    ascending=False
)

,term,token_count,country_count,weight,passes_frequency
3807,urge,697,155,1.0,True
2829,ready,666,152,1.0,True
2830,reaffirm,648,147,1.0,True
919,defend,440,126,1.0,True
1358,exercise,310,118,1.0,True
117,advocate,306,119,1.0,True
2963,reject,267,87,1.0,True
2721,propose,142,64,1.0,True
1927,intend,136,70,1.0,True
899,declare,82,62,1.0,True


In [42]:
frequency_removed = frequency_df[
    ~frequency_df["retain_after_frequency"]
].copy()

print(
    "Terms removed by frequency purification:",
    len(frequency_removed)
)

Terms removed by frequency purification: 59167


In [43]:
frequency_removed[
    frequency_removed["token_count"] > 0
][
    [
        "term",
        "token_count",
        "country_count",
        "weight",
        "sources"
    ]
].sort_values(
    "token_count",
    ascending=False
).head(30)

,term,token_count,country_count,weight,sources
36894,oversee,9,8,0.548155,concept
21149,forgot,9,7,0.505798,concept
54271,stubbornly,9,9,0.536329,concept
1539,all-encompassing,9,7,0.484243,concept
49283,sco,9,3,0.463475,concept
18982,expansive,9,9,0.462428,concept
22108,generalized,9,8,0.517707,concept
48995,satisfying,9,9,0.474990,concept
42437,pulled,9,9,0.459534,concept
26368,indicative,9,8,0.495145,concept


In [44]:
FREQUENCY_PURIFIED_FILE = (
    PURIFICATION_DIR
    / "self_assertion_frequency_purified.csv"
)

frequency_purified.to_csv(
    FREQUENCY_PURIFIED_FILE,
    index=False
)

In [45]:
FREQUENCY_REMOVED_FILE = (
    PURIFICATION_DIR
    / "self_assertion_frequency_removed.csv"
)

frequency_removed.to_csv(
    FREQUENCY_REMOVED_FILE,
    index=False
)

In [46]:
print(FREQUENCY_PURIFIED_FILE)
print(FREQUENCY_REMOVED_FILE)

../data/purification/self_assertion_frequency_purified.csv
../data/purification/self_assertion_frequency_removed.csv


In [47]:
country_total_tokens = {}

for iso3, group in dev_sentences.groupby("iso3"):
    
    total_tokens = 0
    
    for text in group["sentence_text_lower"]:
        total_tokens += len(tokenize_for_kgl(text))
    
    country_total_tokens[iso3] = total_tokens

country_total_tokens = pd.Series(
    country_total_tokens,
    name="total_tokens"
)

print("Number of countries:", len(country_total_tokens))
country_total_tokens.head()

Number of countries: 178


AFG    10597
AGO    17674
ALB    19675
AND    25250
ARE    19221
Name: total_tokens, dtype: int64

In [48]:
retained_terms = set(
    frequency_purified["term"]
)

country_term_counts = defaultdict(Counter)

for row in dev_sentences.itertuples(index=False):
    
    tokens = tokenize_for_kgl(
        row.sentence_text_lower
    )
    
    matched_tokens = [
        token
        for token in tokens
        if token in retained_terms
    ]
    
    if matched_tokens:
        country_term_counts[row.iso3].update(
            matched_tokens
        )

In [49]:
countries = sorted(
    dev_sentences["iso3"].unique()
)

terms = sorted(
    frequency_purified["term"].unique()
)

country_term_matrix = pd.DataFrame(
    0,
    index=countries,
    columns=terms,
    dtype=float
)

for country, counts in country_term_counts.items():
    for term, count in counts.items():
        country_term_matrix.loc[
            country,
            term
        ] = count

In [50]:
print(
    "Country × term matrix shape:",
    country_term_matrix.shape
)

Country × term matrix shape: (178, 3994)


In [51]:
country_term_relative = (
    country_term_matrix
    .div(
        country_total_tokens,
        axis=0
    )
)

In [52]:
country_term_relative.head()

,abandon,abandoned,abandoning,abandonment,abhorrent,abide,abiding,ability,able,abolish,...,wrote,yearn,yet,yield,yielded,yielding,you,your,yourself,zeal
AFG,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.000094,0.000189,0.000377,0.0,...,0.00000,0.0,0.000944,0.000000,0.00000,0.0,0.000283,0.000000,0.0,0.0
AGO,0.000057,0.000057,0.0,0.0,0.000000,0.000000,0.000000,0.000226,0.000453,0.0,...,0.00000,0.0,0.000283,0.000057,0.00000,0.0,0.000679,0.000679,0.0,0.0
ALB,0.000102,0.000000,0.0,0.0,0.000051,0.000000,0.000000,0.000254,0.000203,0.0,...,0.00000,0.0,0.000559,0.000000,0.00000,0.0,0.000508,0.000152,0.0,0.0
AND,0.000040,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000198,0.000515,0.0,...,0.00004,0.0,0.000198,0.000000,0.00004,0.0,0.000475,0.000238,0.0,0.0
ARE,0.000260,0.000052,0.0,0.0,0.000000,0.000156,0.000000,0.000260,0.000416,0.0,...,0.00000,0.0,0.000208,0.000052,0.00000,0.0,0.000208,0.000052,0.0,0.0


In [53]:
weight_lookup = (
    frequency_purified
    .set_index("term")["weight"]
)

In [54]:
country_term_weighted = (
    country_term_relative
    * weight_lookup
)

In [55]:
print(
    "Weighted matrix shape:",
    country_term_weighted.shape
)

Weighted matrix shape: (178, 3994)


In [56]:
from scipy.stats import pearsonr

In [57]:
total_lexicon_score = (
    country_term_weighted.sum(axis=1)
)

total_lexicon_score.head()

AFG    0.353069
AGO    0.354200
ALB    0.363316
AND    0.363005
ARE    0.364011
dtype: float64

In [58]:
correlation_rows = []

for term in country_term_weighted.columns:

    term_vector = country_term_weighted[term]

    rest_of_lexicon = (
        total_lexicon_score
        - term_vector
    )

    # Pearson correlation requires variation
    if (
        term_vector.nunique() <= 1
        or rest_of_lexicon.nunique() <= 1
    ):
        r = np.nan
        p = np.nan

    else:
        r, p = pearsonr(
            term_vector,
            rest_of_lexicon
        )

    correlation_rows.append({
        "term": term,
        "internal_correlation": r,
        "p_value": p
    })

correlation_df = pd.DataFrame(
    correlation_rows
)

In [59]:
correlation_df = correlation_df.merge(
    frequency_purified[
        [
            "term",
            "token_count",
            "country_count",
            "weight",
            "sources",
            "is_seed"
        ]
    ],
    on="term",
    how="left"
)

In [60]:
correlation_df.sort_values(
    "internal_correlation",
    ascending=False
).head(30)

,term,internal_correlation,p_value,token_count,country_count,weight,sources,is_seed
1111,do,0.619580,2.973251e-20,3669,178,0.721730,concept,False
1770,if,0.558233,5.691069e-16,3290,178,0.471666,concept,False
3814,us,0.557071,6.728930e-16,8914,178,0.494046,concept,False
485,can,0.557016,6.782313e-16,6958,178,0.603135,concept,False
1999,it,0.550900,1.620474e-15,20702,178,0.561578,concept,False
3916,what,0.547710,2.534299e-15,2951,177,0.488942,concept,False
385,because,0.541566,5.918687e-15,2334,176,0.512427,concept,False
3612,those,0.516895,1.508166e-13,6921,178,0.542696,concept,False
474,but,0.509006,4.023631e-13,7028,178,0.524259,concept,False
1749,how,0.505025,6.539673e-13,1615,176,0.471774,concept,False


In [61]:
correlation_df.sort_values(
    "internal_correlation",
    ascending=True
).head(30)

,term,internal_correlation,p_value,token_count,country_count,weight,sources,is_seed
3591,the,-0.579500,2.354776e-17,266365,178,0.532953,concept,False
193,and,-0.548594,2.239965e-15,150374,178,0.621598,concept,False
1021,development,-0.367088,4.651903e-07,12868,178,0.483187,concept,False
2623,president,-0.355660,1.105219e-06,6339,178,0.465239,concept,False
3624,through,-0.355127,1.149802e-06,4776,178,0.474777,concept,False
1826,in,-0.338251,3.876688e-06,78537,178,0.468354,concept,False
1720,his,-0.330237,6.737678e-06,5016,178,0.479527,concept,False
2250,movement,-0.329674,6.999958e-06,299,120,0.474400,concept,False
1413,extend,-0.311813,2.266115e-05,531,148,0.662150,concept,False
1593,general,-0.308054,2.874526e-05,5252,178,0.519672,concept,False


In [62]:
correlation_df[
    "internal_correlation"
].describe(
    percentiles=[
        .01,
        .05,
        .10,
        .25,
        .50,
        .75,
        .90,
        .95,
        .99
    ]
)

count    3994.000000
mean        0.010031
std         0.128802
min        -0.579500
1%         -0.242428
5%         -0.169722
10%        -0.137968
25%        -0.074351
50%        -0.004263
75%         0.080135
90%         0.173023
95%         0.248492
99%         0.403537
max         0.619580
Name: internal_correlation, dtype: float64

In [63]:
for threshold in [
    0.00,
    0.05,
    0.10,
    0.15,
    0.20
]:

    n = (
        correlation_df[
            "internal_correlation"
        ] > threshold
    ).sum()

    print(
        f"Threshold > {threshold:.2f}: "
        f"{n} terms"
    )

Threshold > 0.00: 1947 terms
Threshold > 0.05: 1270 terms
Threshold > 0.10: 823 terms
Threshold > 0.15: 489 terms
Threshold > 0.20: 296 terms


In [64]:
seed_correlations = (
    correlation_df[
        correlation_df["is_seed"]
    ]
    [
        [
            "term",
            "internal_correlation",
            "p_value",
            "token_count",
            "country_count",
            "weight"
        ]
    ]
    .sort_values(
        "internal_correlation",
        ascending=False
    )
)

seed_correlations

,term,internal_correlation,p_value,token_count,country_count,weight
919,defend,0.125710,0.094525,440,126,1.0
2829,ready,0.103321,0.169928,666,152,1.0
283,assert,0.069165,0.358944,37,30,1.0
2963,reject,0.069082,0.359522,267,87,1.0
3234,self-assertion,0.046809,0.534959,1,1,1.0
1901,insist,0.041326,0.583890,81,50,1.0
117,advocate,0.037758,0.616807,306,119,1.0
3061,resist,0.011758,0.876218,63,40,1.0
899,declare,-0.003457,0.963467,82,62,1.0
1358,exercise,-0.090446,0.229881,310,118,1.0


In [65]:
correlation_df[
    correlation_df["term"].isin(
        [
            "accept",
            "reject",
            "urge",
            "ready",
            "exercise",
            "self-assertion",
            "assert",
            "defend",
            "implore"
        ]
    )
][
    [
        "term",
        "internal_correlation",
        "token_count",
        "country_count",
        "weight",
        "sources"
    ]
].sort_values(
    "internal_correlation",
    ascending=False
)

,term,internal_correlation,token_count,country_count,weight,sources
24,accept,0.373861,347,125,0.800865,concept; synonym:reject
919,defend,0.125710,440,126,1.000000,concept; seed
2829,ready,0.103321,666,152,1.000000,concept; seed
283,assert,0.069165,37,30,1.000000,concept; seed
2963,reject,0.069082,267,87,1.000000,concept; seed
3234,self-assertion,0.046809,1,1,1.000000,concept; seed
1804,implore,-0.003212,23,19,0.773136,concept; synonym:urge
1358,exercise,-0.090446,310,118,1.000000,concept; seed
3807,urge,-0.128522,697,155,1.000000,concept; seed


In [66]:
country_term_matrix

,abandon,abandoned,abandoning,abandonment,abhorrent,abide,abiding,ability,able,abolish,...,wrote,yearn,yet,yield,yielded,yielding,you,your,yourself,zeal
AFG,0.0,0.0,0.0,0.0,0.0,0.0,1.0,2.0,4.0,0.0,...,0.0,0.0,10.0,0.0,0.0,0.0,3.0,0.0,0.0,0.0
AGO,1.0,1.0,0.0,0.0,0.0,0.0,0.0,4.0,8.0,0.0,...,0.0,0.0,5.0,1.0,0.0,0.0,12.0,12.0,0.0,0.0
ALB,2.0,0.0,0.0,0.0,1.0,0.0,0.0,5.0,4.0,0.0,...,0.0,0.0,11.0,0.0,0.0,0.0,10.0,3.0,0.0,0.0
AND,1.0,0.0,0.0,0.0,0.0,0.0,0.0,5.0,13.0,0.0,...,1.0,0.0,5.0,0.0,1.0,0.0,12.0,6.0,0.0,0.0
ARE,5.0,1.0,0.0,0.0,0.0,3.0,0.0,5.0,8.0,0.0,...,0.0,0.0,4.0,1.0,0.0,0.0,4.0,1.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
WSM,1.0,1.0,0.0,0.0,0.0,0.0,0.0,8.0,5.0,0.0,...,0.0,0.0,13.0,1.0,0.0,0.0,8.0,11.0,0.0,0.0
YEM,0.0,1.0,0.0,0.0,1.0,1.0,0.0,3.0,6.0,1.0,...,0.0,0.0,1.0,1.0,0.0,0.0,28.0,14.0,0.0,0.0
ZAF,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0,10.0,0.0,...,0.0,1.0,9.0,0.0,1.0,0.0,10.0,7.0,0.0,0.0
ZMB,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,7.0,0.0,...,0.0,0.0,3.0,0.0,0.0,0.0,7.0,11.0,0.0,0.0


In [67]:
country_term_weighted_raw = (
    country_term_matrix
    * weight_lookup
)

print(
    "Raw-count weighted matrix shape:",
    country_term_weighted_raw.shape
)

Raw-count weighted matrix shape: (178, 3994)


In [68]:
from scipy.stats import pearsonr

def compute_internal_correlations(weighted_matrix):
    
    total_score = weighted_matrix.sum(axis=1)
    rows = []

    for term in weighted_matrix.columns:

        term_vector = weighted_matrix[term]

        rest_vector = (
            total_score - term_vector
        )

        if (
            term_vector.nunique() <= 1
            or rest_vector.nunique() <= 1
        ):
            r = np.nan
            p = np.nan
        else:
            r, p = pearsonr(
                term_vector,
                rest_vector
            )

        rows.append({
            "term": term,
            "internal_correlation": r,
            "p_value": p
        })

    return pd.DataFrame(rows)

In [69]:
correlation_raw = compute_internal_correlations(
    country_term_weighted_raw
)

correlation_raw = correlation_raw.merge(
    frequency_purified[
        [
            "term",
            "token_count",
            "country_count",
            "weight",
            "sources",
            "is_seed"
        ]
    ],
    on="term",
    how="left"
)

In [70]:
correlation_normalized = (
    correlation_df.copy()
)

In [71]:
correlation_comparison = (
    correlation_raw[
        ["term", "internal_correlation"]
    ]
    .rename(
        columns={
            "internal_correlation":
            "r_raw_count"
        }
    )
    .merge(
        correlation_normalized[
            ["term", "internal_correlation"]
        ].rename(
            columns={
                "internal_correlation":
                "r_token_normalized"
            }
        ),
        on="term",
        how="inner"
    )
)

In [72]:
correlation_comparison["r_difference"] = (
    correlation_comparison["r_token_normalized"]
    - correlation_comparison["r_raw_count"]
)

In [73]:
seed_comparison = (
    correlation_comparison[
        correlation_comparison["term"].isin(
            seed_terms
        )
    ]
    .sort_values(
        "r_token_normalized",
        ascending=False
    )
)

seed_comparison

,term,r_raw_count,r_token_normalized,r_difference
919,defend,0.333292,0.125710,-0.207583
2829,ready,0.230088,0.103321,-0.126767
283,assert,0.091766,0.069165,-0.022600
2963,reject,0.161234,0.069082,-0.092151
3234,self-assertion,0.034861,0.046809,0.011948
1901,insist,0.313993,0.041326,-0.272666
117,advocate,0.181913,0.037758,-0.144155
3061,resist,0.310829,0.011758,-0.299072
899,declare,0.171320,-0.003457,-0.174778
1358,exercise,0.179619,-0.090446,-0.270065


In [74]:
diagnostic_terms = [
    "accept",
    "reject",
    "urge",
    "ready",
    "exercise",
    "assert",
    "defend",
    "propose",
    "reaffirm",
    "advocate",
    "declare",
    "intend",
    "resist",
    "insist",
    "self-assertion",
    "implore"
]

correlation_comparison[
    correlation_comparison["term"].isin(
        diagnostic_terms
    )
].sort_values(
    "r_token_normalized",
    ascending=False
)

,term,r_raw_count,r_token_normalized,r_difference
24,accept,0.404633,0.373861,-0.030772
919,defend,0.333292,0.125710,-0.207583
2829,ready,0.230088,0.103321,-0.126767
283,assert,0.091766,0.069165,-0.022600
2963,reject,0.161234,0.069082,-0.092151
3234,self-assertion,0.034861,0.046809,0.011948
1901,insist,0.313993,0.041326,-0.272666
117,advocate,0.181913,0.037758,-0.144155
3061,resist,0.310829,0.011758,-0.299072
1804,implore,0.085915,-0.003212,-0.089127


In [75]:
for threshold in [
    0.00,
    0.05,
    0.10,
    0.15,
    0.20
]:

    n_raw = (
        correlation_raw[
            "internal_correlation"
        ] > threshold
    ).sum()

    n_normalized = (
        correlation_normalized[
            "internal_correlation"
        ] > threshold
    ).sum()

    print(
        f"Threshold > {threshold:.2f}: "
        f"raw={n_raw}, "
        f"normalized={n_normalized}"
    )

Threshold > 0.00: raw=3635, normalized=1947
Threshold > 0.05: raw=3136, normalized=1270
Threshold > 0.10: raw=2416, normalized=823
Threshold > 0.15: raw=1727, normalized=489
Threshold > 0.20: raw=1183, normalized=296


In [76]:
SYNONYM_FILE = (
    PROJECT_ROOT
    / "data"
    / "expansion"
    / "self_assertion_synonym_expansion_primary.csv"
)

CONCEPT_FILE = (
    PROJECT_ROOT
    / "data"
    / "expansion"
    / "self_assertion_concept_expansion_primary.csv"
)

synonym_expansion_saved = pd.read_csv(SYNONYM_FILE)
concept_expansion_saved = pd.read_csv(CONCEPT_FILE)

print(
    "Synonym rows:",
    len(synonym_expansion_saved)
)

print(
    "Concept rows:",
    len(concept_expansion_saved)
)

Synonym rows: 51
Concept rows: 63791


In [77]:
def build_threshold_lexicon(
    concept_threshold,
    synonym_df,
    concept_df,
    seed_df
):
    
    # ----------------------------
    # Original seeds
    # ----------------------------
    seeds = pd.DataFrame({
        "term": (
            seed_df["seed"]
            .astype(str)
            .str.lower()
        ),
        "weight": 1.0,
        "source": "seed"
    })
    
    # ----------------------------
    # Fixed synonym expansion
    # synonym threshold already = .75
    # ----------------------------
    synonyms = synonym_df.copy()
    
    synonyms["term"] = (
        synonyms["expanded_term"]
        .astype(str)
        .str.lower()
    )
    
    synonyms["weight"] = synonyms["similarity"]
    
    synonyms["source"] = (
        "synonym:" +
        synonyms["seed"].astype(str).str.lower()
    )
    
    synonyms = synonyms[
        ["term", "weight", "source"]
    ]
    
    # ----------------------------
    # Concept expansion
    # apply requested threshold
    # ----------------------------
    concepts = concept_df[
        concept_df["similarity"] >= concept_threshold
    ].copy()
    
    concepts["term"] = (
        concepts["expanded_term"]
        .astype(str)
        .str.lower()
    )
    
    concepts["weight"] = concepts["similarity"]
    concepts["source"] = "concept"
    
    concepts = concepts[
        ["term", "weight", "source"]
    ]
    
    # ----------------------------
    # Union all eligible sources
    # ----------------------------
    combined = pd.concat(
        [
            seeds,
            synonyms,
            concepts
        ],
        ignore_index=True
    )
    
    # Resolve duplicates after lowercasing
    lexicon = (
        combined
        .groupby("term", as_index=False)
        .agg(
            weight=("weight", "max"),
            sources=(
                "source",
                lambda x: "; ".join(
                    sorted(set(x))
                )
            )
        )
    )
    
    return lexicon

In [78]:
test_045 = build_threshold_lexicon(
    concept_threshold=0.45,
    synonym_df=synonym_expansion_saved,
    concept_df=concept_expansion_saved,
    seed_df=seed_df
)

print(
    "Unique terms at .45:",
    len(test_045)
)

Unique terms at .45: 63161


In [79]:
def evaluate_concept_threshold(
    concept_threshold,
    synonym_df,
    concept_df,
    seed_df,
    frequency_df,
    country_term_relative
):
    
    # ------------------------------------
    # 1. Build threshold-specific lexicon
    # ------------------------------------
    lexicon = build_threshold_lexicon(
        concept_threshold=concept_threshold,
        synonym_df=synonym_df,
        concept_df=concept_df,
        seed_df=seed_df
    )
    
    # ------------------------------------
    # 2. Add corpus-frequency diagnostics
    # ------------------------------------
    count_columns = [
        "term",
        "token_count",
        "sentence_count",
        "speech_count",
        "country_count",
        "country_year_count"
    ]
    
    lexicon = lexicon.merge(
        frequency_df[count_columns],
        on="term",
        how="left"
    )
    
    # Any unmatched term means zero corpus occurrences
    count_vars = [
        "token_count",
        "sentence_count",
        "speech_count",
        "country_count",
        "country_year_count"
    ]
    
    lexicon[count_vars] = (
        lexicon[count_vars]
        .fillna(0)
    )
    
    # ------------------------------------
    # 3. Identify seeds
    # ------------------------------------
    local_seed_terms = set(
        seed_df["seed"]
        .astype(str)
        .str.lower()
    )
    
    lexicon["is_seed"] = (
        lexicon["term"]
        .isin(local_seed_terms)
    )
    
    # ------------------------------------
    # 4. Frequency purification
    # ------------------------------------
    lexicon["passes_frequency"] = (
        lexicon["token_count"] >= 10
    )
    
    frequency_kept = lexicon[
        lexicon["passes_frequency"]
        | lexicon["is_seed"]
    ].copy()
    
    # ------------------------------------
    # 5. Keep terms represented in our
    # country-frequency matrix
    # ------------------------------------
    terms = [
        term
        for term in frequency_kept["term"]
        if term in country_term_relative.columns
    ]
    
    frequency_kept = (
        frequency_kept[
            frequency_kept["term"].isin(terms)
        ]
        .copy()
    )
    
    # ------------------------------------
    # 6. Threshold-specific weights
    # ------------------------------------
    weights = (
        frequency_kept
        .set_index("term")["weight"]
    )
    
    weighted_matrix = (
        country_term_relative[terms]
        * weights
    )
    
    # ------------------------------------
    # 7. Internal correlations
    # ------------------------------------
    correlations = (
        compute_internal_correlations(
            weighted_matrix
        )
    )
    
    correlations = correlations.merge(
        frequency_kept[
            [
                "term",
                "token_count",
                "country_count",
                "weight",
                "sources",
                "is_seed"
            ]
        ],
        on="term",
        how="left"
    )
    
    return {
        "concept_threshold":
            concept_threshold,
        
        "raw_lexicon":
            lexicon,
        
        "frequency_kept":
            frequency_kept,
        
        "weighted_matrix":
            weighted_matrix,
        
        "correlations":
            correlations
    }

In [80]:
CONCEPT_THRESHOLDS = [
    0.45,
    0.50,
    0.55,
    0.60,
    0.65,
    0.70
]

threshold_results = {}

for threshold in CONCEPT_THRESHOLDS:
    
    print(
        f"Running threshold {threshold:.2f}..."
    )
    
    threshold_results[threshold] = (
        evaluate_concept_threshold(
            concept_threshold=threshold,
            synonym_df=synonym_expansion_saved,
            concept_df=concept_expansion_saved,
            seed_df=seed_df,
            frequency_df=frequency_df,
            country_term_relative=country_term_relative
        )
    )

print("Finished.")

Running threshold 0.45...
Running threshold 0.50...
Running threshold 0.55...
Running threshold 0.60...
Running threshold 0.65...
Running threshold 0.70...
Finished.


In [81]:
threshold_summary_rows = []

for threshold, result in threshold_results.items():
    
    correlations = result["correlations"]
    
    threshold_summary_rows.append({
        "concept_threshold":
            threshold,
        
        "raw_union_terms":
            len(result["raw_lexicon"]),
        
        "after_frequency":
            len(result["frequency_kept"]),
        
        "r_gt_0":
            (
                correlations[
                    "internal_correlation"
                ] > 0
            ).sum(),
        
        "r_gt_05":
            (
                correlations[
                    "internal_correlation"
                ] > 0.05
            ).sum(),
        
        "r_gt_10":
            (
                correlations[
                    "internal_correlation"
                ] > 0.10
            ).sum(),
        
        "r_gt_15":
            (
                correlations[
                    "internal_correlation"
                ] > 0.15
            ).sum(),
        
        "r_gt_20":
            (
                correlations[
                    "internal_correlation"
                ] > 0.20
            ).sum()
    })

threshold_summary = pd.DataFrame(
    threshold_summary_rows
)

threshold_summary

,concept_threshold,raw_union_terms,after_frequency,r_gt_0,r_gt_05,r_gt_10,r_gt_15,r_gt_20
0,0.45,63161,3994,1947,1270,823,489,296
1,0.50,28439,2225,1255,908,595,395,243
2,0.55,10701,1171,677,506,348,258,181
3,0.60,3452,543,359,267,177,115,75
4,0.65,874,225,151,115,77,51,34
5,0.70,188,88,69,56,35,22,14


In [82]:
seed_threshold_rows = []

for threshold, result in threshold_results.items():
    
    correlations = result["correlations"]
    
    seed_rows = correlations[
        correlations["is_seed"]
    ]
    
    for _, row in seed_rows.iterrows():
        
        seed_threshold_rows.append({
            "concept_threshold":
                threshold,
            
            "term":
                row["term"],
            
            "internal_correlation":
                row["internal_correlation"]
        })

seed_threshold_df = pd.DataFrame(
    seed_threshold_rows
)

In [83]:
seed_threshold_pivot = (
    seed_threshold_df
    .pivot(
        index="term",
        columns="concept_threshold",
        values="internal_correlation"
    )
)

seed_threshold_pivot

concept_threshold,0.45,0.50,0.55,0.60,0.65,0.70
term,,,,,,
advocate,0.037758,-0.018805,0.034801,-0.015947,0.076553,0.137438
assert,0.069165,0.111919,0.075342,0.141635,0.091455,0.223964
declare,-0.003457,0.100743,0.078648,0.120314,0.145206,0.215620
defend,0.125710,0.188880,0.149818,0.140379,0.168793,0.103338
exercise,-0.090446,-0.062965,-0.115379,-0.093355,-0.073203,0.045026
insist,0.041326,0.075417,0.062693,0.073979,-0.012335,0.044812
intend,-0.114470,-0.135301,-0.144169,-0.132681,0.020179,0.088131
propose,-0.154816,-0.177672,-0.194597,-0.183716,-0.003610,0.045043
ready,0.103321,0.062680,0.102737,0.023295,0.251293,0.082463


In [84]:
accept_rows = []

for threshold, result in threshold_results.items():
    
    correlations = result["correlations"]
    
    row = correlations[
        correlations["term"] == "accept"
    ]
    
    if len(row) == 1:
        
        accept_rows.append({
            "concept_threshold":
                threshold,
            
            "internal_correlation":
                row.iloc[0][
                    "internal_correlation"
                ],
            
            "weight":
                row.iloc[0]["weight"],
            
            "sources":
                row.iloc[0]["sources"]
        })

accept_threshold_df = pd.DataFrame(
    accept_rows
)

accept_threshold_df

,concept_threshold,internal_correlation,weight,sources
0,0.45,0.373861,0.800865,concept; synonym:reject
1,0.50,0.426671,0.800865,concept; synonym:reject
2,0.55,0.415403,0.800865,concept; synonym:reject
3,0.60,0.393418,0.800865,concept; synonym:reject
4,0.65,0.345041,0.800865,concept; synonym:reject
5,0.70,0.338443,0.800865,concept; synonym:reject


In [85]:
threshold_summary.to_csv(
    PURIFICATION_DIR
    / "self_assertion_concept_threshold_summary.csv",
    index=False
)

seed_threshold_pivot.to_csv(
    PURIFICATION_DIR
    / "self_assertion_seed_threshold_sensitivity.csv"
)

accept_threshold_df.to_csv(
    PURIFICATION_DIR
    / "self_assertion_accept_threshold_sensitivity.csv",
    index=False
)

In [86]:
INDEPENDENT_AUDIT_FILE = (
    PROJECT_ROOT
    / "data"
    / "seed_audit"
    / "Self Assertion Independent Coverage Audit_CODED.csv"
)

independent_audit = pd.read_csv(
    INDEPENDENT_AUDIT_FILE
)

print(
    "Rows:",
    len(independent_audit)
)

print(
    "\nColumns:"
)

print(
    independent_audit.columns.tolist()
)

Rows: 250

Columns:
['iso3', 'year', 'filename', 'sentence_id', 'previous_sentence', 'sentence_text', 'next_sentence', 'is_self_assertion', 'facet', 'key_expression', 'covered_by_current_seed_family', 'coverage_notes']


In [87]:
independent_audit.head()

,iso3,year,filename,sentence_id,previous_sentence,sentence_text,next_sentence,is_self_assertion,facet,key_expression,covered_by_current_seed_family,coverage_notes
0,YEM,2020,YEM_75_2020.txt,31,What is even worse is that this militia took t...,What you see today is a ruthless escalation by...,It attacks residential areas with ballistic mi...,no,NaN,NaN,NaN,No focal-national agentive self-positioning in...
1,GRC,2022,GRC_77_2022.txt,51,It is also the only NATO country not implement...,"When it comes to Greece, Tiirkiye’s leadership...",Their language is increasingly bellicose.,no,NaN,NaN,NaN,No focal-national agentive self-positioning in...
2,SGP,2020,SGP_75_2020.txt,71,We have a responsibility to help our people to...,We also have to help our businesses leverage d...,"At the international level, governments, busin...",yes,commit_intended_action,have to help our businesses,no,Focal actor states an intended responsibility/...
3,MCO,2023,MCO_78_2023.txt,49,What major countries have achieved in space sh...,While we look for the presence of water on the...,The young people of our nations place before u...,no,NaN,NaN,NaN,No focal-national agentive self-positioning in...
4,DZA,2023,DZA_78_2023.txt,76,my country issued its fourth national report i...,My country is firmly convinced that respect fo...,That is why we work to strengthen human rights...,no,NaN,NaN,NaN,No focal-national agentive self-positioning in...


In [88]:
for col in independent_audit.columns:
    if independent_audit[col].nunique(
        dropna=False
    ) <= 10:
        
        print(
            f"\n--- {col} ---"
        )
        
        print(
            independent_audit[col]
            .value_counts(
                dropna=False
            )
        )


--- year ---
year
2025    39
2022    35
2023    27
2017    25
2021    24
2024    24
2016    21
2018    20
2019    18
2020    17
Name: count, dtype: int64

--- is_self_assertion ---
is_self_assertion
no           181
yes           66
ambiguous      3
Name: count, dtype: int64

--- facet ---
facet
NaN                         181
commit_intended_action       20
advance_outcome              16
influence_outcome            10
exercise_capability_role      9
maintain_position             5
resist_constraint             4
claim_position                3
defend_self_relevant          2
Name: count, dtype: int64

--- covered_by_current_seed_family ---
covered_by_current_seed_family
NaN         184
no           28
probably     21
yes          17
Name: count, dtype: int64


In [89]:
validation_df = independent_audit[
    independent_audit["is_self_assertion"].isin(
        ["yes", "no"]
    )
].copy()

validation_df["gold_label"] = (
    validation_df["is_self_assertion"] == "yes"
)

print("Validation sentences:", len(validation_df))
print(validation_df["gold_label"].value_counts())

Validation sentences: 247
gold_label
False    181
True      66
Name: count, dtype: int64


In [90]:
def sentence_has_lexicon_hit(text, lexicon_terms):
    
    tokens = tokenize_for_kgl(text)
    
    return any(
        token in lexicon_terms
        for token in tokens
    )

In [91]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

validation_rows = []

for threshold, result in threshold_results.items():
    
    lexicon_terms_threshold = set(
        result["frequency_kept"]["term"]
    )
    
    predictions = validation_df[
        "sentence_text"
    ].apply(
        lambda text: sentence_has_lexicon_hit(
            text,
            lexicon_terms_threshold
        )
    )
    
    gold = validation_df["gold_label"]
    
    tn, fp, fn, tp = confusion_matrix(
        gold,
        predictions,
        labels=[False, True]
    ).ravel()
    
    validation_rows.append({
        "concept_threshold": threshold,
        "n_terms_after_frequency":
            len(lexicon_terms_threshold),
        "true_positive": tp,
        "false_positive": fp,
        "false_negative": fn,
        "true_negative": tn,
        "precision": precision_score(
            gold,
            predictions,
            zero_division=0
        ),
        "recall": recall_score(
            gold,
            predictions,
            zero_division=0
        ),
        "f1": f1_score(
            gold,
            predictions,
            zero_division=0
        )
    })

retrieval_validation = pd.DataFrame(
    validation_rows
)

retrieval_validation

,concept_threshold,n_terms_after_frequency,true_positive,false_positive,false_negative,true_negative,precision,recall,f1
0,0.45,3994,66,181,0,0,0.267206,1.000000,0.421725
1,0.50,2225,66,180,0,1,0.268293,1.000000,0.423077
2,0.55,1171,66,173,0,8,0.276151,1.000000,0.432787
3,0.60,543,63,162,3,19,0.280000,0.954545,0.432990
4,0.65,225,46,71,20,110,0.393162,0.696970,0.502732
5,0.70,88,29,19,37,162,0.604167,0.439394,0.508772


In [92]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

PURIFICATION_THRESHOLDS = [
    0.00,
    0.05,
    0.10,
    0.15,
    0.20
]

joint_validation_rows = []

for concept_threshold, result in threshold_results.items():

    correlations = result["correlations"]

    for purification_threshold in PURIFICATION_THRESHOLDS:

        # Keep only terms whose normalized
        # internal correlation exceeds threshold
        retained_terms = set(
            correlations.loc[
                correlations["internal_correlation"]
                > purification_threshold,
                "term"
            ]
        )

        predictions = validation_df[
            "sentence_text"
        ].apply(
            lambda text: sentence_has_lexicon_hit(
                text,
                retained_terms
            )
        )

        gold = validation_df["gold_label"]

        tn, fp, fn, tp = confusion_matrix(
            gold,
            predictions,
            labels=[False, True]
        ).ravel()

        precision = precision_score(
            gold,
            predictions,
            zero_division=0
        )

        recall = recall_score(
            gold,
            predictions,
            zero_division=0
        )

        f1 = f1_score(
            gold,
            predictions,
            zero_division=0
        )

        # F2 weights recall more heavily than precision.
        # Useful because this KGL is a retrieval layer.
        if precision == 0 and recall == 0:
            f2 = 0
        else:
            f2 = (
                5 * precision * recall
                / (4 * precision + recall)
            )

        joint_validation_rows.append({
            "concept_threshold":
                concept_threshold,

            "purification_threshold":
                purification_threshold,

            "n_terms":
                len(retained_terms),

            "true_positive":
                tp,

            "false_positive":
                fp,

            "false_negative":
                fn,

            "true_negative":
                tn,

            "precision":
                precision,

            "recall":
                recall,

            "f1":
                f1,

            "f2":
                f2
        })

joint_validation = pd.DataFrame(
    joint_validation_rows
)

In [93]:
joint_validation.sort_values(
    [
        "concept_threshold",
        "purification_threshold"
    ]
)

,concept_threshold,purification_threshold,n_terms,true_positive,false_positive,false_negative,true_negative,precision,recall,f1,f2
0,0.45,0.00,1947,66,176,0,5,0.272727,1.000000,0.428571,0.652174
1,0.45,0.05,1270,65,175,1,6,0.270833,0.984848,0.424837,0.644841
2,0.45,0.10,823,63,168,3,13,0.272727,0.954545,0.424242,0.636364
3,0.45,0.15,489,60,166,6,15,0.265487,0.909091,0.410959,0.612245
4,0.45,0.20,296,60,161,6,20,0.271493,0.909091,0.418118,0.618557
5,0.50,0.00,1255,64,171,2,10,0.272340,0.969697,0.425249,0.641283
6,0.50,0.05,908,64,168,2,13,0.275862,0.969697,0.429530,0.645161
7,0.50,0.10,595,62,161,4,20,0.278027,0.939394,0.429066,0.636550
8,0.50,0.15,395,58,150,8,31,0.278846,0.878788,0.423358,0.614407
9,0.50,0.20,243,56,144,10,37,0.280000,0.848485,0.421053,0.603448


In [94]:
high_recall_configs = (
    joint_validation[
        joint_validation["recall"] >= 0.80
    ]
    .sort_values(
        [
            "f2",
            "precision"
        ],
        ascending=False
    )
)

high_recall_configs

,concept_threshold,purification_threshold,n_terms,true_positive,false_positive,false_negative,true_negative,precision,recall,f1,f2
0,0.45,0.00,1947,66,176,0,5,0.272727,1.000000,0.428571,0.652174
6,0.50,0.05,908,64,168,2,13,0.275862,0.969697,0.429530,0.645161
1,0.45,0.05,1270,65,175,1,6,0.270833,0.984848,0.424837,0.644841
5,0.50,0.00,1255,64,171,2,10,0.272340,0.969697,0.425249,0.641283
10,0.55,0.00,677,62,160,4,21,0.279279,0.939394,0.430556,0.637860
7,0.50,0.10,595,62,161,4,20,0.278027,0.939394,0.429066,0.636550
2,0.45,0.10,823,63,168,3,13,0.272727,0.954545,0.424242,0.636364
15,0.60,0.00,359,59,143,7,38,0.292079,0.893939,0.440299,0.633047
16,0.60,0.05,267,59,143,7,38,0.292079,0.893939,0.440299,0.633047
11,0.55,0.05,506,59,152,7,29,0.279621,0.893939,0.425993,0.621053


In [95]:
joint_validation.sort_values(
    "f2",
    ascending=False
).head(15)

,concept_threshold,purification_threshold,n_terms,true_positive,false_positive,false_negative,true_negative,precision,recall,f1,f2
0,0.45,0.00,1947,66,176,0,5,0.272727,1.000000,0.428571,0.652174
6,0.50,0.05,908,64,168,2,13,0.275862,0.969697,0.429530,0.645161
1,0.45,0.05,1270,65,175,1,6,0.270833,0.984848,0.424837,0.644841
5,0.50,0.00,1255,64,171,2,10,0.272340,0.969697,0.425249,0.641283
10,0.55,0.00,677,62,160,4,21,0.279279,0.939394,0.430556,0.637860
7,0.50,0.10,595,62,161,4,20,0.278027,0.939394,0.429066,0.636550
2,0.45,0.10,823,63,168,3,13,0.272727,0.954545,0.424242,0.636364
16,0.60,0.05,267,59,143,7,38,0.292079,0.893939,0.440299,0.633047
15,0.60,0.00,359,59,143,7,38,0.292079,0.893939,0.440299,0.633047
11,0.55,0.05,506,59,152,7,29,0.279621,0.893939,0.425993,0.621053


In [96]:
joint_validation[
    joint_validation["concept_threshold"].isin(
        [0.60, 0.65, 0.70]
    )
].sort_values(
    [
        "concept_threshold",
        "purification_threshold"
    ]
)

,concept_threshold,purification_threshold,n_terms,true_positive,false_positive,false_negative,true_negative,precision,recall,f1,f2
15,0.60,0.00,359,59,143,7,38,0.292079,0.893939,0.440299,0.633047
16,0.60,0.05,267,59,143,7,38,0.292079,0.893939,0.440299,0.633047
17,0.60,0.10,177,48,121,18,60,0.284024,0.727273,0.408511,0.554273
18,0.60,0.15,115,46,119,20,62,0.278788,0.696970,0.398268,0.536131
19,0.60,0.20,75,41,105,25,76,0.280822,0.621212,0.386792,0.500000
20,0.65,0.00,151,39,60,27,121,0.393939,0.590909,0.472727,0.537190
21,0.65,0.05,115,36,58,30,123,0.382979,0.545455,0.450000,0.502793
22,0.65,0.10,77,30,57,36,124,0.344828,0.454545,0.392157,0.427350
23,0.65,0.15,51,29,53,37,128,0.353659,0.439394,0.391892,0.419075
24,0.65,0.20,34,29,48,37,133,0.376623,0.439394,0.405594,0.425220


In [97]:
JOINT_VALIDATION_FILE = (
    PURIFICATION_DIR
    / "self_assertion_joint_threshold_validation.csv"
)

joint_validation.to_csv(
    JOINT_VALIDATION_FILE,
    index=False
)

print(JOINT_VALIDATION_FILE)

HIGH_RECALL_FILE = (
    PURIFICATION_DIR
    / "self_assertion_high_recall_configs.csv"
)

high_recall_configs.to_csv(
    HIGH_RECALL_FILE,
    index=False
)

print(HIGH_RECALL_FILE)

FOCUSED_CONFIG_FILE = (
    PURIFICATION_DIR
    / "self_assertion_focused_threshold_configs.csv"
)

joint_validation[
    joint_validation["concept_threshold"].isin(
        [0.60, 0.65, 0.70]
    )
].sort_values(
    [
        "concept_threshold",
        "purification_threshold"
    ]
).to_csv(
    FOCUSED_CONFIG_FILE,
    index=False
)

print(FOCUSED_CONFIG_FILE)

../data/purification/self_assertion_joint_threshold_validation.csv
../data/purification/self_assertion_high_recall_configs.csv
../data/purification/self_assertion_focused_threshold_configs.csv


In [98]:
def evaluate_term_set(term_set, label):
    
    predictions = validation_df[
        "sentence_text"
    ].apply(
        lambda text: sentence_has_lexicon_hit(
            text,
            term_set
        )
    )
    
    gold = validation_df["gold_label"]
    
    tn, fp, fn, tp = confusion_matrix(
        gold,
        predictions,
        labels=[False, True]
    ).ravel()
    
    precision = precision_score(
        gold,
        predictions,
        zero_division=0
    )
    
    recall = recall_score(
        gold,
        predictions,
        zero_division=0
    )
    
    f1 = f1_score(
        gold,
        predictions,
        zero_division=0
    )
    
    return {
        "model": label,
        "n_terms": len(term_set),
        "true_positive": tp,
        "false_positive": fp,
        "false_negative": fn,
        "true_negative": tn,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

In [99]:
seed_only_terms = set(
    seed_df["seed"]
    .astype(str)
    .str.lower()
)

seed_only_result = evaluate_term_set(
    seed_only_terms,
    "seed_only"
)

seed_only_result

{'model': 'seed_only',
 'n_terms': 14,
 'true_positive': np.int64(12),
 'false_positive': np.int64(3),
 'false_negative': np.int64(54),
 'true_negative': np.int64(178),
 'precision': 0.8,
 'recall': 0.18181818181818182,
 'f1': 0.2962962962962963}

In [100]:
synonym_terms = set(
    synonym_expansion_saved[
        "expanded_term"
    ]
    .astype(str)
    .str.lower()
)

seed_plus_synonym_terms = (
    seed_only_terms
    | synonym_terms
)

seed_plus_synonym_result = evaluate_term_set(
    seed_plus_synonym_terms,
    "seed_plus_synonym"
)

seed_plus_synonym_resultsynonym_terms = set(
    synonym_expansion_saved[
        "expanded_term"
    ]
    .astype(str)
    .str.lower()
)

seed_plus_synonym_terms = (
    seed_only_terms
    | synonym_terms
)

seed_plus_synonym_result = evaluate_term_set(
    seed_plus_synonym_terms,
    "seed_plus_synonym"
)

seed_plus_synonym_result

{'model': 'seed_plus_synonym',
 'n_terms': 65,
 'true_positive': np.int64(13),
 'false_positive': np.int64(4),
 'false_negative': np.int64(53),
 'true_negative': np.int64(177),
 'precision': 0.7647058823529411,
 'recall': 0.19696969696969696,
 'f1': 0.3132530120481928}

In [101]:
baseline_comparison = pd.DataFrame(
    [
        seed_only_result,
        seed_plus_synonym_result
    ]
)

baseline_comparison

,model,n_terms,true_positive,false_positive,false_negative,true_negative,precision,recall,f1
0,seed_only,14,12,3,54,178,0.800000,0.181818,0.296296
1,seed_plus_synonym,65,13,4,53,177,0.764706,0.196970,0.313253


In [102]:
terms_060 = set(
    threshold_results[0.60][
        "frequency_kept"
    ]["term"]
)

In [103]:
validation_060 = validation_df.copy()

validation_060["lexical_hit_060"] = (
    validation_060["sentence_text"]
    .apply(
        lambda text: sentence_has_lexicon_hit(
            text,
            terms_060
        )
    )
)

In [104]:
false_positives_060 = validation_060[
    (validation_060["lexical_hit_060"])
    & (~validation_060["gold_label"])
].copy()

print(
    "False positives at .60:",
    len(false_positives_060)
)

False positives at .60: 162


In [105]:
false_negatives_060 = validation_060[
    (~validation_060["lexical_hit_060"])
    & (validation_060["gold_label"])
].copy()

print(
    "False negatives at .60:",
    len(false_negatives_060)
)

False negatives at .60: 3


In [106]:
def get_lexicon_hits(text, lexicon_terms):
    
    tokens = tokenize_for_kgl(text)
    
    hits = sorted(
        set(
            token
            for token in tokens
            if token in lexicon_terms
        )
    )
    
    return "; ".join(hits)

In [107]:
validation_060["lexicon_hits_060"] = (
    validation_060["sentence_text"]
    .apply(
        lambda text: get_lexicon_hits(
            text,
            terms_060
        )
    )
)

In [108]:
false_positives_060 = validation_060[
    (validation_060["lexical_hit_060"])
    & (~validation_060["gold_label"])
].copy()

false_negatives_060 = validation_060[
    (~validation_060["lexical_hit_060"])
    & (validation_060["gold_label"])
].copy()

In [109]:
FALSE_POSITIVE_FILE = (
    PURIFICATION_DIR
    / "self_assertion_060_false_positives.csv"
)

false_positives_060[
    [
        "iso3",
        "year",
        "filename",
        "sentence_id",
        "previous_sentence",
        "sentence_text",
        "next_sentence",
        "lexicon_hits_060",
        "coverage_notes"
    ]
].to_csv(
    FALSE_POSITIVE_FILE,
    index=False
)

In [110]:
FALSE_NEGATIVE_FILE = (
    PURIFICATION_DIR
    / "self_assertion_060_false_negatives.csv"
)

false_negatives_060[
    [
        "iso3",
        "year",
        "filename",
        "sentence_id",
        "previous_sentence",
        "sentence_text",
        "next_sentence",
        "facet",
        "key_expression",
        "coverage_notes"
    ]
].to_csv(
    FALSE_NEGATIVE_FILE,
    index=False
)

In [111]:
from spacy.lang.en.stop_words import STOP_WORDS

print("Number of spaCy English stopwords:", len(STOP_WORDS))

Number of spaCy English stopwords: 326


In [112]:
frequency_kept_060 = (
    threshold_results[0.60]["frequency_kept"]
    .copy()
)

frequency_kept_060["is_stopword"] = (
    frequency_kept_060["term"].isin(STOP_WORDS)
)

stopword_terms_060 = (
    frequency_kept_060[
        frequency_kept_060["is_stopword"]
    ]
    .sort_values(
        "token_count",
        ascending=False
    )
)

print(
    "Stopwords in the .60 candidate lexicon:",
    len(stopword_terms_060)
)

stopword_terms_060[
    [
        "term",
        "token_count",
        "country_count",
        "weight",
        "sources",
        "is_seed"
    ]
].head(100)

Stopwords in the .60 candidate lexicon: 42


,term,token_count,country_count,weight,sources,is_seed
156,and,150374,178,0.621598,concept,False
3265,to,124839,178,0.625660,concept,False
3234,that,55148,178,0.636330,concept,False
1419,have,19375,178,0.620219,concept,False
130,all,16384,178,0.607538,concept,False
3426,will,14488,178,0.668819,concept,False
1832,not,12908,178,0.638071,concept,False
1795,must,10139,178,0.671552,concept,False
376,can,6958,178,0.603135,concept,False
3248,they,5657,178,0.600620,concept,False


In [113]:
seed_stopwords = (
    seed_terms
    & set(STOP_WORDS)
)

print("Seed terms classified as stopwords:")
print(seed_stopwords)

Seed terms classified as stopwords:
set()


In [114]:
HARD_FUNCTION_WORDS = {
    "and",
    "to",
    "that",
    "have",
    "all",
    "will",
    "not",
    "must",
    "can",
    "they",
    "them",
    "would",
    "should",
    "cannot",
    "therefore",
    "both",
    "still",
    "always",
    "never",
    "could",
    "does",
    "indeed",
    "might",
    "either",
    "really",
    "otherwise",
    "please"
}

In [115]:
candidate_060_filtered = (
    threshold_results[0.60]["frequency_kept"]
    .loc[
        lambda df:
        ~df["term"].isin(HARD_FUNCTION_WORDS)
    ]
    .copy()
)

print(
    "Terms before hard-function filtering:",
    len(
        threshold_results[0.60][
            "frequency_kept"
        ]
    )
)

print(
    "Terms after hard-function filtering:",
    len(candidate_060_filtered)
)

Terms before hard-function filtering: 543
Terms after hard-function filtering: 516


In [116]:
terms_060_filtered = set(
    candidate_060_filtered["term"]
)

In [117]:
filtered_060_result = evaluate_term_set(
    terms_060_filtered,
    "concept_060_after_function_filter"
)

filtered_060_result

{'model': 'concept_060_after_function_filter',
 'n_terms': 516,
 'true_positive': np.int64(51),
 'false_positive': np.int64(99),
 'false_negative': np.int64(15),
 'true_negative': np.int64(82),
 'precision': 0.34,
 'recall': 0.7727272727272727,
 'f1': 0.4722222222222222}

In [118]:
validation_060_filtered = (
    validation_df.copy()
)

validation_060_filtered[
    "lexical_hit_060_filtered"
] = (
    validation_060_filtered[
        "sentence_text"
    ]
    .apply(
        lambda text:
        sentence_has_lexicon_hit(
            text,
            terms_060_filtered
        )
    )
)

In [119]:
false_positives_060_filtered = (
    validation_060_filtered[
        (
            validation_060_filtered[
                "lexical_hit_060_filtered"
            ]
        )
        &
        (
            ~validation_060_filtered[
                "gold_label"
            ]
        )
    ]
    .copy()
)

false_negatives_060_filtered = (
    validation_060_filtered[
        (
            ~validation_060_filtered[
                "lexical_hit_060_filtered"
            ]
        )
        &
        (
            validation_060_filtered[
                "gold_label"
            ]
        )
    ]
    .copy()
)

print(
    "Remaining false positives:",
    len(false_positives_060_filtered)
)

print(
    "False negatives:",
    len(false_negatives_060_filtered)
)

Remaining false positives: 99
False negatives: 15


In [120]:
false_positives_060_filtered[
    "lexicon_hits"
] = (
    false_positives_060_filtered[
        "sentence_text"
    ]
    .apply(
        lambda text:
        get_lexicon_hits(
            text,
            terms_060_filtered
        )
    )
)

In [121]:
remaining_fp_hit_counts = (
    false_positives_060_filtered[
        "lexicon_hits"
    ]
    .str.split("; ")
    .explode()
    .value_counts()
)

remaining_fp_hit_counts.head(50)

lexicon_hits
need           6
change         5
know           4
make           4
even           4
use            3
hope           3
take           3
support        3
ensure         3
allow          3
continue       3
just           3
remain         3
let            3
implement      3
respect        2
go             2
challenge      2
view           2
express        2
require        2
strengthen     2
conclude       2
act            2
put            2
provide        2
reverse        2
fight          2
serve          2
move           2
believe        2
extend         2
threaten       2
respond        2
see            1
convinced      1
undermine      1
appropriate    1
seek           1
force          1
assertion      1
generate       1
find           1
ultimately     1
negotiate      1
say            1
struggle       1
call           1
leave          1
Name: count, dtype: int64

In [122]:
FALSE_NEGATIVE_FILTERED_FILE = (
    PURIFICATION_DIR
    / "self_assertion_060_filtered_false_negatives.csv"
)

false_negatives_060_filtered[
    [
        "iso3",
        "year",
        "filename",
        "sentence_id",
        "previous_sentence",
        "sentence_text",
        "next_sentence",
        "facet",
        "key_expression",
        "coverage_notes"
    ]
].to_csv(
    FALSE_NEGATIVE_FILTERED_FILE,
    index=False
)

print(FALSE_NEGATIVE_FILTERED_FILE)

../data/purification/self_assertion_060_filtered_false_negatives.csv


In [123]:
remaining_fp_hit_counts_df = (
    remaining_fp_hit_counts
    .rename_axis("term")
    .reset_index(name="false_positive_sentence_count")
)

FP_HIT_COUNTS_FILE = (
    PURIFICATION_DIR
    / "self_assertion_060_filtered_fp_hit_counts.csv"
)

remaining_fp_hit_counts_df.to_csv(
    FP_HIT_COUNTS_FILE,
    index=False
)

print(FP_HIT_COUNTS_FILE)

../data/purification/self_assertion_060_filtered_fp_hit_counts.csv


In [124]:
validation_context = validation_df.copy()

validation_context["context_text"] = (
    validation_context[
        [
            "previous_sentence",
            "sentence_text",
            "next_sentence"
        ]
    ]
    .fillna("")
    .agg(" ".join, axis=1)
)

In [125]:
validation_context[
    [
        "previous_sentence",
        "sentence_text",
        "next_sentence",
        "context_text"
    ]
].head()

,previous_sentence,sentence_text,next_sentence,context_text
0,What is even worse is that this militia took t...,What you see today is a ruthless escalation by...,It attacks residential areas with ballistic mi...,What is even worse is that this militia took t...
1,It is also the only NATO country not implement...,"When it comes to Greece, Tiirkiye’s leadership...",Their language is increasingly bellicose.,It is also the only NATO country not implement...
2,We have a responsibility to help our people to...,We also have to help our businesses leverage d...,"At the international level, governments, busin...",We have a responsibility to help our people to...
3,What major countries have achieved in space sh...,While we look for the presence of water on the...,The young people of our nations place before u...,What major countries have achieved in space sh...
4,my country issued its fourth national report i...,My country is firmly convinced that respect fo...,That is why we work to strengthen human rights...,my country issued its fourth national report i...


In [126]:
context_060_result = evaluate_term_set(
    terms_060_filtered,
    "concept_060_function_filtered_context"
)

In [127]:
def evaluate_term_set_on_context(
    dataframe,
    term_set,
    label
):
    
    predictions = dataframe[
        "context_text"
    ].apply(
        lambda text: sentence_has_lexicon_hit(
            text,
            term_set
        )
    )
    
    gold = dataframe["gold_label"]
    
    tn, fp, fn, tp = confusion_matrix(
        gold,
        predictions,
        labels=[False, True]
    ).ravel()
    
    precision = precision_score(
        gold,
        predictions,
        zero_division=0
    )
    
    recall = recall_score(
        gold,
        predictions,
        zero_division=0
    )
    
    f1 = f1_score(
        gold,
        predictions,
        zero_division=0
    )
    
    return {
        "model": label,
        "n_terms": len(term_set),
        "true_positive": tp,
        "false_positive": fp,
        "false_negative": fn,
        "true_negative": tn,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

In [128]:
context_060_result = evaluate_term_set_on_context(
    validation_context,
    terms_060_filtered,
    "concept_060_function_filtered_context"
)

context_060_result

{'model': 'concept_060_function_filtered_context',
 'n_terms': 516,
 'true_positive': np.int64(63),
 'false_positive': np.int64(159),
 'false_negative': np.int64(3),
 'true_negative': np.int64(22),
 'precision': 0.28378378378378377,
 'recall': 0.9545454545454546,
 'f1': 0.4375}

In [129]:
validation_context[
    "lexical_hit_060_filtered_context"
] = (
    validation_context[
        "context_text"
    ]
    .apply(
        lambda text:
        sentence_has_lexicon_hit(
            text,
            terms_060_filtered
        )
    )
)

In [130]:
false_positives_context_060 = (
    validation_context[
        (
            validation_context[
                "lexical_hit_060_filtered_context"
            ]
        )
        &
        (
            ~validation_context["gold_label"]
        )
    ]
    .copy()
)

false_negatives_context_060 = (
    validation_context[
        (
            ~validation_context[
                "lexical_hit_060_filtered_context"
            ]
        )
        &
        (
            validation_context["gold_label"]
        )
    ]
    .copy()
)

print(
    "Context-window false positives:",
    len(false_positives_context_060)
)

print(
    "Context-window false negatives:",
    len(false_negatives_context_060)
)

Context-window false positives: 159
Context-window false negatives: 3


In [131]:
false_positives_context_060[
    "lexicon_hits"
] = (
    false_positives_context_060[
        "context_text"
    ]
    .apply(
        lambda text: get_lexicon_hits(
            text,
            terms_060_filtered
        )
    )
)

In [132]:
context_fp_hit_counts = (
    false_positives_context_060[
        "lexicon_hits"
    ]
    .str.split("; ")
    .explode()
    .value_counts()
    .rename_axis("term")
    .reset_index(
        name="false_positive_context_count"
    )
)

context_fp_hit_counts.head(50)

,term,false_positive_context_count
0,support,18
1,need,14
2,ensure,13
3,change,13
4,even,11
5,work,10
6,do,10
7,see,9
8,respect,9
9,just,8


In [133]:
context_fp_hit_counts = (
    context_fp_hit_counts
    .merge(
        candidate_060_filtered[
            [
                "term",
                "weight",
                "sources",
                "token_count",
                "country_count",
                "is_seed"
            ]
        ],
        on="term",
        how="left"
    )
)

context_fp_hit_counts.head(50)

,term,false_positive_context_count,weight,sources,token_count,country_count,is_seed
0,support,18,0.709948,concept,6844,178,False
1,need,14,0.686426,concept,4728,178,False
2,ensure,13,0.616068,concept,2913,178,False
3,change,13,0.636577,concept,6197,178,False
4,even,11,0.638988,concept,2148,176,False
5,work,10,0.608292,concept,4157,178,False
6,do,10,0.721730,concept,3669,178,False
7,see,9,0.614783,concept,1739,174,False
8,respect,9,0.639559,concept,2673,178,False
9,just,8,0.630096,concept,2675,178,False


In [134]:
CONTEXT_FP_TERM_FILE = (
    PURIFICATION_DIR
    / "self_assertion_060_context_fp_term_counts.csv"
)

context_fp_hit_counts.to_csv(
    CONTEXT_FP_TERM_FILE,
    index=False
)

print(CONTEXT_FP_TERM_FILE)

../data/purification/self_assertion_060_context_fp_term_counts.csv


In [135]:
CONTEXT_FP_CASE_FILE = (
    PURIFICATION_DIR
    / "self_assertion_060_context_false_positives.csv"
)

false_positives_context_060[
    [
        "iso3",
        "year",
        "filename",
        "sentence_id",
        "previous_sentence",
        "sentence_text",
        "next_sentence",
        "lexicon_hits",
        "coverage_notes"
    ]
].to_csv(
    CONTEXT_FP_CASE_FILE,
    index=False
)

print(CONTEXT_FP_CASE_FILE)

../data/purification/self_assertion_060_context_false_positives.csv


In [136]:
validation_context["lexicon_hits_060_filtered"] = (
    validation_context["context_text"]
    .apply(
        lambda text: get_lexicon_hits(
            text,
            terms_060_filtered
        )
    )
)

In [137]:
term_context_long = (
    validation_context[
        [
            "gold_label",
            "lexicon_hits_060_filtered"
        ]
    ]
    .assign(
        term=lambda df:
        df["lexicon_hits_060_filtered"]
        .str.split("; ")
    )
    .explode("term")
)

In [138]:
term_context_long = (
    term_context_long[
        term_context_long["term"].notna()
        & (term_context_long["term"] != "")
    ]
    .copy()
)

In [139]:
term_validation_counts = (
    term_context_long
    .groupby("term")
    .agg(
        true_positive_context_count=(
            "gold_label",
            "sum"
        ),
        total_hit_context_count=(
            "gold_label",
            "size"
        )
    )
    .reset_index()
)

term_validation_counts[
    "false_positive_context_count"
] = (
    term_validation_counts[
        "total_hit_context_count"
    ]
    -
    term_validation_counts[
        "true_positive_context_count"
    ]
)

In [140]:
term_validation_counts[
    "context_precision"
] = (
    term_validation_counts[
        "true_positive_context_count"
    ]
    /
    term_validation_counts[
        "total_hit_context_count"
    ]
)

In [141]:
term_validation_counts = (
    term_validation_counts
    .merge(
        candidate_060_filtered[
            [
                "term",
                "weight",
                "sources",
                "token_count",
                "country_count",
                "is_seed"
            ]
        ],
        on="term",
        how="left"
    )
)

In [142]:
term_validation_counts = (
    term_validation_counts
    .sort_values(
        [
            "total_hit_context_count",
            "context_precision"
        ],
        ascending=[False, True]
    )
)

term_validation_counts.head(50)

,term,true_positive_context_count,total_hit_context_count,false_positive_context_count,context_precision,weight,sources,token_count,country_count,is_seed
223,support,13,31,18,0.419355,0.709948,concept,6844,178,False
141,need,6,20,14,0.300000,0.686426,concept,4728,178,False
84,ensure,5,18,13,0.277778,0.616068,concept,2913,178,False
42,change,2,15,13,0.133333,0.636577,concept,6197,178,False
244,work,4,14,10,0.285714,0.608292,concept,4157,178,False
194,respect,5,14,9,0.357143,0.639559,concept,2673,178,False
87,even,2,13,11,0.153846,0.638988,concept,2148,176,False
57,continue,5,13,8,0.384615,0.708292,concept,4009,178,False
226,take,5,13,8,0.384615,0.668233,concept,2747,178,False
76,do,2,12,10,0.166667,0.721730,concept,3669,178,False


In [143]:
TERM_VALIDATION_FILE = (
    PURIFICATION_DIR
    / "self_assertion_060_term_validation_counts.csv"
)

term_validation_counts.to_csv(
    TERM_VALIDATION_FILE,
    index=False
)

print(TERM_VALIDATION_FILE)

../data/purification/self_assertion_060_term_validation_counts.csv


In [144]:
empirical_pruning = (
    candidate_060_filtered.copy()
    .merge(
        term_validation_counts[
            [
                "term",
                "true_positive_context_count",
                "false_positive_context_count",
                "total_hit_context_count",
                "context_precision"
            ]
        ],
        on="term",
        how="left"
    )
)

In [145]:
count_cols = [
    "true_positive_context_count",
    "false_positive_context_count",
    "total_hit_context_count"
]

empirical_pruning[count_cols] = (
    empirical_pruning[count_cols]
    .fillna(0)
    .astype(int)
)

In [146]:
empirical_pruning["context_precision"] = (
    empirical_pruning["context_precision"]
    .astype(float)
)

In [147]:
def empirical_pruning_decision(row):
    
    # Theory-derived seeds are always retained
    if row["is_seed"]:
        return "retain_seed"
    
    n = row["total_hit_context_count"]
    tp = row["true_positive_context_count"]
    fp = row["false_positive_context_count"]
    
    # Too little validation evidence to judge
    if n < 3:
        return "retain_insufficient_evidence"
    
    # Empirically more often associated with true SA
    if tp > fp:
        return "retain_empirical"
    
    # Enough evidence, but not more TP than FP
    return "exclude_empirical"

In [148]:
empirical_pruning["pruning_decision"] = (
    empirical_pruning.apply(
        empirical_pruning_decision,
        axis=1
    )
)

In [149]:
empirical_pruning[
    "pruning_decision"
].value_counts()

pruning_decision
retain_insufficient_evidence    412
exclude_empirical                75
retain_empirical                 15
retain_seed                      14
Name: count, dtype: int64

In [150]:
empirically_retained = (
    empirical_pruning[
        empirical_pruning[
            "pruning_decision"
        ].str.startswith("retain")
    ]
    .copy()
)

empirically_excluded = (
    empirical_pruning[
        empirical_pruning[
            "pruning_decision"
        ] == "exclude_empirical"
    ]
    .copy()
)

print(
    "Terms before empirical pruning:",
    len(empirical_pruning)
)

print(
    "Terms retained:",
    len(empirically_retained)
)

print(
    "Terms excluded:",
    len(empirically_excluded)
)

Terms before empirical pruning: 516
Terms retained: 441
Terms excluded: 75


In [151]:
retained_terms_empirical = set(
    empirically_retained["term"]
)

missing_seeds = (
    seed_terms
    - retained_terms_empirical
)

print(
    "Frozen seeds missing after pruning:",
    missing_seeds
)

Frozen seeds missing after pruning: set()


In [152]:
empirical_pruned_result = (
    evaluate_term_set_on_context(
        validation_context,
        retained_terms_empirical,
        "concept_060_empirically_pruned"
    )
)

empirical_pruned_result

{'model': 'concept_060_empirically_pruned',
 'n_terms': 441,
 'true_positive': np.int64(52),
 'false_positive': np.int64(94),
 'false_negative': np.int64(14),
 'true_negative': np.int64(87),
 'precision': 0.3561643835616438,
 'recall': 0.7878787878787878,
 'f1': 0.49056603773584906}

In [153]:
EMPIRICAL_RETAINED_FILE = (
    PURIFICATION_DIR
    / "self_assertion_060_empirically_retained.csv"
)

EMPIRICAL_EXCLUDED_FILE = (
    PURIFICATION_DIR
    / "self_assertion_060_empirically_excluded.csv"
)

empirically_retained.to_csv(
    EMPIRICAL_RETAINED_FILE,
    index=False
)

empirically_excluded.to_csv(
    EMPIRICAL_EXCLUDED_FILE,
    index=False
)

print(EMPIRICAL_RETAINED_FILE)
print(EMPIRICAL_EXCLUDED_FILE)

../data/purification/self_assertion_060_empirically_retained.csv
../data/purification/self_assertion_060_empirically_excluded.csv


In [154]:
empirical_pruned_result = (
    evaluate_term_set_on_context(
        validation_context,
        retained_terms_empirical,
        "concept_060_empirically_pruned"
    )
)

empirical_pruned_result

{'model': 'concept_060_empirically_pruned',
 'n_terms': 441,
 'true_positive': np.int64(52),
 'false_positive': np.int64(94),
 'false_negative': np.int64(14),
 'true_negative': np.int64(87),
 'precision': 0.3561643835616438,
 'recall': 0.7878787878787878,
 'f1': 0.49056603773584906}

In [155]:
def conservative_pruning_decision(row):

    # Frozen theory seeds are always retained
    if row["is_seed"]:
        return "retain_seed"

    n = row["total_hit_context_count"]

    # Too little evidence for empirical exclusion
    if n < 5:
        return "retain_insufficient_evidence"

    precision = row["context_precision"]

    # Exclude only strongly noise-dominated terms
    if precision <= 0.25:
        return "exclude_empirical"

    return "retain_empirical"

In [156]:
conservative_pruning = empirical_pruning.copy()

conservative_pruning["pruning_decision_v2"] = (
    conservative_pruning.apply(
        conservative_pruning_decision,
        axis=1
    )
)

conservative_pruning[
    "pruning_decision_v2"
].value_counts()

pruning_decision_v2
retain_insufficient_evidence    456
retain_empirical                 28
exclude_empirical                18
retain_seed                      14
Name: count, dtype: int64

In [157]:
conservative_retained = (
    conservative_pruning[
        conservative_pruning[
            "pruning_decision_v2"
        ].str.startswith("retain")
    ]
    .copy()
)

retained_terms_conservative = set(
    conservative_retained["term"]
)

print(
    "Terms before pruning:",
    len(conservative_pruning)
)

print(
    "Terms retained:",
    len(conservative_retained)
)

print(
    "Terms excluded:",
    len(conservative_pruning)
    - len(conservative_retained)
)

Terms before pruning: 516
Terms retained: 498
Terms excluded: 18


In [158]:
print(
    "Missing frozen seeds:",
    seed_terms - retained_terms_conservative
)

Missing frozen seeds: set()


In [159]:
conservative_pruned_result = (
    evaluate_term_set_on_context(
        validation_context,
        retained_terms_conservative,
        "concept_060_conservative_empirical_pruning"
    )
)

conservative_pruned_result

{'model': 'concept_060_conservative_empirical_pruning',
 'n_terms': 498,
 'true_positive': np.int64(62),
 'false_positive': np.int64(147),
 'false_negative': np.int64(4),
 'true_negative': np.int64(34),
 'precision': 0.2966507177033493,
 'recall': 0.9393939393939394,
 'f1': 0.4509090909090909}

In [160]:
retained_terms_conservative

{'abandon',
 'abide',
 'able',
 'accede',
 'accept',
 'accepted',
 'accompany',
 'accomplish',
 'accuse',
 'achieve',
 'acknowledge',
 'acquire',
 'activate',
 'actively',
 'actually',
 'adapt',
 'add',
 'adhere',
 'admit',
 'adopt',
 'advance',
 'advise',
 'advocate',
 'advocated',
 'advocates',
 'advocating',
 'affirm',
 'agree',
 'agreed',
 'agreeing',
 'agrees',
 'aim',
 'announce',
 'anticipate',
 'apologize',
 'appeal',
 'applaud',
 'apply',
 'appreciate',
 'approach',
 'appropriate',
 'approve',
 'argue',
 'ask',
 'asking',
 'aspire',
 'assert',
 'asserted',
 'asserting',
 'assertion',
 'assess',
 'assist',
 'associate',
 'assume',
 'assure',
 'assured',
 'attempt',
 'attempting',
 'attribute',
 'authorize',
 'avail',
 'avoid',
 'await',
 'begin',
 'believing',
 'betray',
 'boldly',
 'bring',
 'build',
 'buy',
 'call',
 'carry',
 'cause',
 'cease',
 'celebrate',
 'certain',
 'challenge',
 'challenged',
 'characterize',
 'choose',
 'chose',
 'cite',
 'claim',
 'claiming',
 'clari

In [161]:
len(retained_terms_conservative)

498

In [162]:
from collections import Counter
import pandas as pd
import numpy as np

In [163]:
DEV_SENTENCE_FILE = (
    PROJECT_ROOT
    / "data"
    / "preprocessed"
    / "development_sentences_clean.csv"
)

development_clean = pd.read_csv(
    DEV_SENTENCE_FILE
)

print(development_clean.shape)
print(development_clean["iso3"].nunique())

(157346, 9)
178


In [164]:
retained_term_set = set(
    retained_terms_conservative
)

country_total_tokens = Counter()
country_term_counts = {}

for iso3, group in development_clean.groupby("iso3"):
    
    total_counter = Counter()
    
    for text in group["sentence_text_lower"].fillna(""):
        tokens = tokenize_for_kgl(text)
        total_counter.update(tokens)
    
    country_total_tokens[iso3] = sum(
        total_counter.values()
    )
    
    country_term_counts[iso3] = {
        term: total_counter.get(term, 0)
        for term in retained_term_set
    }

In [165]:
country_term_matrix = (
    pd.DataFrame
    .from_dict(
        country_term_counts,
        orient="index"
    )
    .sort_index()
    .fillna(0)
)

country_term_matrix.shape

(178, 498)

In [166]:
print(
    "Countries:",
    country_term_matrix.shape[0]
)

print(
    "Terms:",
    country_term_matrix.shape[1]
)

Countries: 178
Terms: 498


In [167]:
country_token_series = pd.Series(
    country_total_tokens
).reindex(
    country_term_matrix.index
)

normalized_term_matrix = (
    country_term_matrix
    .div(
        country_token_series,
        axis=0
    )
)

In [168]:
normalized_term_matrix.shape

(178, 498)

In [169]:
term_weights = (
    conservative_retained
    .set_index("term")["weight"]
    .reindex(
        country_term_matrix.columns
    )
)

In [170]:
print(
    "Missing weights:",
    term_weights.isna().sum()
)

Missing weights: 0


In [171]:
weighted_term_matrix = (
    normalized_term_matrix
    .mul(
        term_weights,
        axis=1
    )
)

In [172]:
total_weighted_score = (
    weighted_term_matrix.sum(axis=1)
)

correlation_rows = []

for term in weighted_term_matrix.columns:
    
    term_values = (
        weighted_term_matrix[term]
    )
    
    rest_values = (
        total_weighted_score
        - term_values
    )
    
    if (
        term_values.nunique() <= 1
        or rest_values.nunique() <= 1
    ):
        r = np.nan
        
    else:
        r = term_values.corr(
            rest_values,
            method="pearson"
        )
    
    correlation_rows.append({
        "term": term,
        "internal_correlation": r
    })

correlations_498 = pd.DataFrame(
    correlation_rows
)

In [173]:
correlations_498 = (
    correlations_498
    .merge(
        conservative_retained,
        on="term",
        how="left"
    )
)

In [174]:
correlations_498[
    "internal_correlation"
].describe(
    percentiles=[
        .01,
        .05,
        .10,
        .25,
        .50,
        .75,
        .90,
        .95,
        .99
    ]
)

count    498.000000
mean       0.033698
std        0.099972
min       -0.284640
1%        -0.191329
5%        -0.147785
10%       -0.090110
25%       -0.027220
50%        0.035362
75%        0.104779
90%        0.160877
95%        0.193526
99%        0.256289
max        0.297706
Name: internal_correlation, dtype: float64

In [175]:
for threshold in [
    0.00,
    0.05,
    0.10,
    0.15,
    0.20
]:
    
    n_survive = (
        correlations_498[
            "internal_correlation"
        ] > threshold
    ).sum()
    
    print(
        f"r > {threshold:.2f}: "
        f"{n_survive} terms"
    )

r > 0.00: 323 terms
r > 0.05: 217 terms
r > 0.10: 131 terms
r > 0.15: 68 terms
r > 0.20: 21 terms


In [176]:
correlations_498[
    correlations_498["is_seed"]
][
    [
        "term",
        "internal_correlation",
        "token_count",
        "country_count"
    ]
].sort_values(
    "internal_correlation",
    ascending=False
)

,term,internal_correlation,token_count,country_count
129,reject,0.220990,267,87
190,declare,0.123958,82,62
192,assert,0.108689,37,30
92,resist,0.090307,63,40
46,reaffirm,0.088878,648,147
336,defend,0.082413,440,126
435,ready,0.076863,666,152
472,urge,0.074149,697,155
389,advocate,0.015934,306,119
8,intend,-0.019100,136,70


In [177]:
CORRELATION_THRESHOLDS = [
    0.00,
    0.05,
    0.10,
    0.15,
    0.20
]

correlation_validation_rows = []

for threshold in CORRELATION_THRESHOLDS:
    
    # Terms that pass internal-correlation purification
    passing_terms = set(
        correlations_498.loc[
            correlations_498["internal_correlation"] > threshold,
            "term"
        ]
    )
    
    # Project adaptation:
    # always protect the 14 theory-derived seeds
    final_terms = (
        passing_terms
        | seed_terms
    )
    
    result = evaluate_term_set_on_context(
        validation_context,
        final_terms,
        f"correlation_r_gt_{threshold:.2f}"
    )
    
    result["correlation_threshold"] = threshold
    result["n_correlation_passing"] = len(passing_terms)
    result["n_final_with_seed_protection"] = len(final_terms)
    
    correlation_validation_rows.append(
        result
    )

correlation_validation = pd.DataFrame(
    correlation_validation_rows
)

correlation_validation[
    [
        "correlation_threshold",
        "n_correlation_passing",
        "n_final_with_seed_protection",
        "true_positive",
        "false_positive",
        "false_negative",
        "true_negative",
        "precision",
        "recall",
        "f1"
    ]
]

,correlation_threshold,n_correlation_passing,n_final_with_seed_protection,true_positive,false_positive,false_negative,true_negative,precision,recall,f1
0,0.00,323,328,60,133,6,48,0.310881,0.909091,0.463320
1,0.05,217,223,56,120,10,61,0.318182,0.848485,0.462810
2,0.10,131,142,53,106,13,75,0.333333,0.803030,0.471111
3,0.15,68,81,50,81,16,100,0.381679,0.757576,0.507614
4,0.20,21,34,37,50,29,131,0.425287,0.560606,0.483660


In [178]:
correlation_validation["f2"] = (
    5
    * correlation_validation["precision"]
    * correlation_validation["recall"]
    /
    (
        4 * correlation_validation["precision"]
        + correlation_validation["recall"]
    )
)

correlation_validation[
    [
        "correlation_threshold",
        "n_final_with_seed_protection",
        "precision",
        "recall",
        "f1",
        "f2"
    ]
]

,correlation_threshold,n_final_with_seed_protection,precision,recall,f1,f2
0,0.00,328,0.310881,0.909091,0.463320,0.656455
1,0.05,223,0.318182,0.848485,0.462810,0.636364
2,0.10,142,0.333333,0.803030,0.471111,0.626478
3,0.15,81,0.381679,0.757576,0.507614,0.632911
4,0.20,34,0.425287,0.560606,0.483660,0.527066


In [179]:
CORRELATION_VALIDATION_FILE = (
    PURIFICATION_DIR
    / "self_assertion_correlation_threshold_validation.csv"
)

correlation_validation.to_csv(
    CORRELATION_VALIDATION_FILE,
    index=False
)

print(CORRELATION_VALIDATION_FILE)

../data/purification/self_assertion_correlation_threshold_validation.csv


In [180]:
CORRELATION_THRESHOLD_FINAL = 0.00

correlation_purified = (
    correlations_498[
        (
            correlations_498["internal_correlation"]
            > CORRELATION_THRESHOLD_FINAL
        )
        |
        (
            correlations_498["is_seed"]
        )
    ]
    .copy()
)


In [182]:
print(
    "Final candidate KGL terms:",
    len(correlation_purified)
)

print(
    "Protected seeds retained:",
    correlation_purified.loc[
        correlation_purified["is_seed"],
        "term"
    ].nunique()
)

print(
    "Missing frozen seeds:",
    seed_terms
    - set(correlation_purified["term"])
)

Final candidate KGL terms: 328
Protected seeds retained: 14
Missing frozen seeds: set()


In [183]:
FINAL_CANDIDATE_KGL_FILE = (
    PURIFICATION_DIR
    / "self_assertion_kgl_candidate_final.csv"
)

correlation_purified.to_csv(
    FINAL_CANDIDATE_KGL_FILE,
    index=False
)

print(FINAL_CANDIDATE_KGL_FILE)

../data/purification/self_assertion_kgl_candidate_final.csv


In [184]:
from pathlib import Path
import pandas as pd
import numpy as np
import re

PROJECT_ROOT = Path("..")

CANDIDATE_FILE = (
    PROJECT_ROOT
    / "data"
    / "purification"
    / "self_assertion_kgl_candidate_final.csv"
)

DEV_FILE = (
    PROJECT_ROOT
    / "data"
    / "preprocessed"
    / "development_sentences_clean.csv"
)

candidate_kgl = pd.read_csv(CANDIDATE_FILE)
dev = pd.read_csv(DEV_FILE)

print("Candidate terms:", len(candidate_kgl))
print("Development sentences:", len(dev))

Candidate terms: 328
Development sentences: 157346


In [186]:
dev = (
    dev
    .sort_values(["filename", "sentence_id"])
    .reset_index(drop=True)
)

dev["previous_sentence"] = (
    dev.groupby("filename")["sentence_text"].shift(1)
)

dev["next_sentence"] = (
    dev.groupby("filename")["sentence_text"].shift(-1)
)

dev["previous_sentence"] = (
    dev["previous_sentence"].fillna("")
)

dev["next_sentence"] = (
    dev["next_sentence"].fillna("")
)

In [187]:
TOKEN_PATTERN = re.compile(r"[a-z]+(?:-[a-z]+)*")

def tokenize_for_kgl(text):
    return TOKEN_PATTERN.findall(str(text).lower())

dev["token_set"] = (
    dev["sentence_text_lower"]
    .fillna("")
    .apply(lambda x: set(tokenize_for_kgl(x)))
)

In [189]:
RANDOM_SEED = 20260921
N_CONTEXTS_PER_TERM = 5

audit_rows = []

for _, term_info in candidate_kgl.iterrows():

    term = term_info["term"]

    # Find every development-corpus sentence containing this term
    hits = dev[
        dev["token_set"].apply(lambda tokens: term in tokens)
    ].copy()

    if len(hits) == 0:
        print(f"WARNING: no occurrences found for {term}")
        continue

    # ---------------------------------------------------------
    # Prefer examples from different speeches.
    #
    # Randomly shuffle all hits, then keep the first occurrence
    # from each speech. This avoids groupby.apply(), which caused
    # the filename KeyError under the current pandas version.
    # ---------------------------------------------------------

    shuffled_hits = hits.sample(
        frac=1,
        random_state=RANDOM_SEED
    )

    one_per_speech = (
        shuffled_hits
        .drop_duplicates(
            subset="filename",
            keep="first"
        )
    )

    # If there are at least 5 different speeches,
    # take 5 examples from different speeches.
    if len(one_per_speech) >= N_CONTEXTS_PER_TERM:

        sampled = one_per_speech.head(
            N_CONTEXTS_PER_TERM
        ).copy()

    else:

        # Start with all available different-speech examples.
        sampled = one_per_speech.copy()

        remaining_n = (
            N_CONTEXTS_PER_TERM
            - len(sampled)
        )

        # Identify sentence occurrences already selected.
        selected_keys = set(
            zip(
                sampled["filename"],
                sampled["sentence_id"]
            )
        )

        # Find other occurrences that were not already selected.
        remaining_hits = hits[
            ~hits.apply(
                lambda row:
                    (
                        row["filename"],
                        row["sentence_id"]
                    )
                    in selected_keys,
                axis=1
            )
        ].copy()

        # Add extra occurrences if necessary.
        if remaining_n > 0 and len(remaining_hits) > 0:

            extra = remaining_hits.sample(
                n=min(
                    remaining_n,
                    len(remaining_hits)
                ),
                random_state=RANDOM_SEED
            )

            sampled = pd.concat(
                [sampled, extra],
                ignore_index=True
            )

    sampled = sampled.reset_index(drop=True)

    # ---------------------------------------------------------
    # Store the sampled examples
    # ---------------------------------------------------------

    for example_number, (_, row) in enumerate(
        sampled.iterrows(),
        start=1
    ):

        audit_rows.append({

            # Candidate-term information
            "term": term,
            "is_seed": term_info["is_seed"],
            "sources": term_info["sources"],
            "weight": term_info["weight"],
            "internal_correlation":
                term_info["internal_correlation"],
            "token_count":
                term_info["token_count"],

            # Corpus occurrence information
            "example_number": example_number,
            "iso3": row["iso3"],
            "year": row["year"],
            "filename": row["filename"],
            "sentence_id": row["sentence_id"],

            # ±1 sentence context
            "previous_sentence":
                row["previous_sentence"],

            "target_sentence":
                row["sentence_text"],

            "next_sentence":
                row["next_sentence"],

            # Fields for later coding
            "term_is_self_assertion_indicator": "",
            "coding_reason": ""
        })


semantic_audit = pd.DataFrame(audit_rows)

print("Terms represented:", semantic_audit["term"].nunique())
print("Total sampled contexts:", len(semantic_audit))

print("\nNumber of contexts sampled per term:")
print(
    semantic_audit
    .groupby("term")
    .size()
    .value_counts()
    .sort_index()
)

Terms represented: 328
Total sampled contexts: 1636

Number of contexts sampled per term:
1      1
5    327
Name: count, dtype: int64


In [190]:
term_review_rows = []

for term, group in semantic_audit.groupby(
    "term",
    sort=True
):

    group = group.sort_values(
        "example_number"
    )

    first = group.iloc[0]

    output = {
        "term": term,
        "is_seed": first["is_seed"],
        "sources": first["sources"],
        "weight": first["weight"],
        "internal_correlation":
            first["internal_correlation"],
        "token_count":
            first["token_count"],
        "n_contexts_sampled":
            len(group),
    }

    for _, row in group.iterrows():

        n = int(row["example_number"])

        output[f"context_{n}"] = (
            f"[PREVIOUS] {row['previous_sentence']} "
            f"[TARGET] {row['target_sentence']} "
            f"[NEXT] {row['next_sentence']}"
        )

    output["semantic_decision"] = ""
    output["decision_reason"] = ""

    term_review_rows.append(output)


term_review = pd.DataFrame(
    term_review_rows
)

print("Rows in term-level review sheet:", len(term_review))
print(
    "Distribution of sampled contexts per term:"
)
print(
    term_review["n_contexts_sampled"]
    .value_counts()
    .sort_index()
)

term_review.head()

Rows in term-level review sheet: 328
Distribution of sampled contexts per term:
n_contexts_sampled
1      1
5    327
Name: count, dtype: int64


,term,is_seed,sources,weight,internal_correlation,token_count,n_contexts_sampled,context_1,context_2,context_3,context_4,context_5,semantic_decision,decision_reason
0,abandon,False,concept,0.664300,0.155117,130,5,[PREVIOUS] We must redouble our efforts to ful...,[PREVIOUS] This is especially alarming for sma...,"[PREVIOUS] Yet, we have constantly faced the r...",[PREVIOUS] Latin America firmly rejects any at...,[PREVIOUS] Nuclear proliferation must remain a...,,
1,able,False,concept,0.648534,0.206989,1345,5,"[PREVIOUS] Unfortunately, that is no longer th...",[PREVIOUS] We support the activities of the Sp...,[PREVIOUS] Heatwaves have swept across Europe ...,"[PREVIOUS] By virtue of this right, the people...","[PREVIOUS] This is a rare opportunity, support...",,
2,accede,False,concept,0.653675,0.029758,15,5,[PREVIOUS] We will join the World Health Organ...,[PREVIOUS] The g7+ is an intergovernmental pla...,[PREVIOUS] Recent events should be creating fa...,[PREVIOUS] The Organization shoulders a major ...,[PREVIOUS] It gives me pleasure to announce th...,,
3,accept,False,concept; synonym:reject,0.800865,0.237863,347,5,[PREVIOUS] That is simply a recipe for more co...,"[PREVIOUS] Dehumanized, murdered. [TARGET] Jus...","[PREVIOUS] We have our differing views, but it...","[PREVIOUS] More than 3,000 cases of abduction,...","[PREVIOUS] Only eight months later, I led the ...",,
4,accompany,False,concept,0.626890,0.005513,33,5,[PREVIOUS] We believe that the inclusion of th...,[PREVIOUS] We hope that smooth elections in th...,[PREVIOUS] A future characterized by the rapid...,"[PREVIOUS] Finally, may God bless the free wor...",[PREVIOUS] This means that the issue of Africa...,,


In [191]:
REVIEW_FILE = (
    PROJECT_ROOT
    / "data"
    / "purification"
    / "self_assertion_semantic_term_review.csv"
)

term_review.to_csv(
    REVIEW_FILE,
    index=False
)

print("Saved to:")
print(REVIEW_FILE)

Saved to:
../data/purification/self_assertion_semantic_term_review.csv


In [194]:
reviewed_terms = pd.read_csv(
    PROJECT_ROOT
    / "data"
    / "purification"
    / "self_assertion_semantic_term_review.csv"
)

# Make these columns able to store text labels
reviewed_terms["semantic_decision"] = (
    reviewed_terms["semantic_decision"]
    .astype("object")
)

reviewed_terms["decision_reason"] = (
    reviewed_terms["decision_reason"]
    .astype("object")
)

In [195]:
MANUAL_KEEP_FROM_UNCERTAIN = {
    "call",
    "confirm",
    "decide",
    "decided",
    "favor",
    "favour",
    "recommend",
    "request",
    "requested",
    "withdraw",
}

reviewed_terms.loc[
    reviewed_terms["term"].isin(MANUAL_KEEP_FROM_UNCERTAIN),
    "semantic_decision"
] = "KEEP"

reviewed_terms.loc[
    reviewed_terms["term"].isin(MANUAL_KEEP_FROM_UNCERTAIN),
    "decision_reason"
] = (
    "Retained after manual contextual adjudication of previously uncertain terms."
)

In [197]:
reviewed_terms = pd.read_csv(
    PROJECT_ROOT
    / "data"
    / "purification"
    / "self_assertion_semantic_term_review.csv"
)

# Allow text labels in these columns
reviewed_terms["semantic_decision"] = (
    reviewed_terms["semantic_decision"]
    .astype("object")
)

reviewed_terms["decision_reason"] = (
    reviewed_terms["decision_reason"]
    .astype("object")
)

In [198]:
FIRST_PASS_KEEP = {
    "advocate","affirm","announce","appeal","argue","assert","asserted","asserting","assure",
    "clarify","commit","condemn",
    "declare","declares","declaring",
    "defend","defended","defending","defy","demand","denounce","deplore","disagree",
    "emphasise","emphasize","endorse",
    "insist","insists","intend","intends",
    "proclaim","propose","proposes","proposing","pursue",
    "ready","reaffirm","reaffirmed","reassert","recommit",
    "refuse","refusing","reiterate",
    "reject","rejected","rejecting","rejects",
    "renounce","resist","resisted","resisting","restate",
    "self-assertion","strive","submit","uphold","urge","urged","warn"
}

FIRST_PASS_UNCERTAIN = {
    "accede","call","claim","claiming","confirm","decide","decided",
    "encourage","enact","exercise","favor","favour",
    "negotiate","promise","ratify","recommend","request","requested",
    "seek","seeking","support","supported","withdraw"
}

In [199]:
def assign_first_pass(term):
    if term in FIRST_PASS_KEEP:
        return "KEEP"
    elif term in FIRST_PASS_UNCERTAIN:
        return "UNCERTAIN"
    else:
        return "EXCLUDE"

reviewed_terms["semantic_decision"] = (
    reviewed_terms["term"]
    .apply(assign_first_pass)
)

In [200]:
MANUAL_KEEP_FROM_UNCERTAIN = {
    "call",
    "confirm",
    "decide",
    "decided",
    "favor",
    "favour",
    "recommend",
    "request",
    "requested",
    "withdraw",
}

reviewed_terms.loc[
    reviewed_terms["term"].isin(MANUAL_KEEP_FROM_UNCERTAIN),
    "semantic_decision"
] = "KEEP"

In [201]:
reviewed_terms.loc[
    reviewed_terms["semantic_decision"].eq("UNCERTAIN"),
    "semantic_decision"
] = "EXCLUDE"

In [202]:
print(
    reviewed_terms["semantic_decision"]
    .value_counts()
)

semantic_decision
EXCLUDE    259
KEEP        69
Name: count, dtype: int64


In [203]:
FINAL_SEMANTIC_REVIEW_FILE = (
    PROJECT_ROOT
    / "data"
    / "purification"
    / "self_assertion_semantic_term_review_FINAL.csv"
)

reviewed_terms.to_csv(
    FINAL_SEMANTIC_REVIEW_FILE,
    index=False
)

print(FINAL_SEMANTIC_REVIEW_FILE)

../data/purification/self_assertion_semantic_term_review_FINAL.csv


In [204]:
final_semantic_review = pd.read_csv(
    PROJECT_ROOT
    / "data"
    / "purification"
    / "self_assertion_semantic_term_review_FINAL.csv"
)

semantic_keep_terms = set(
    final_semantic_review.loc[
        final_semantic_review["semantic_decision"].eq("KEEP"),
        "term"
    ]
)

semantic_purified_kgl = (
    candidate_kgl[
        candidate_kgl["term"].isin(semantic_keep_terms)
    ]
    .copy()
    .reset_index(drop=True)
)

print("Semantically retained terms:", len(semantic_purified_kgl))

print(
    "Retained seeds:",
    semantic_purified_kgl["is_seed"].sum()
)

print(
    "Terms missing from candidate file:",
    semantic_keep_terms
    - set(candidate_kgl["term"])
)

Semantically retained terms: 69
Retained seeds: 13
Terms missing from candidate file: set()


In [205]:
from collections import Counter, defaultdict
import numpy as np
from scipy.stats import pearsonr

In [206]:
retained_terms_69 = set(
    semantic_purified_kgl["term"]
)

country_term_counts = defaultdict(Counter)
country_total_tokens = Counter()

for _, row in dev.iterrows():

    country = row["iso3"]

    tokens = tokenize_for_kgl(
        row["sentence_text_lower"]
    )

    country_total_tokens[country] += len(tokens)

    relevant_tokens = [
        token
        for token in tokens
        if token in retained_terms_69
    ]

    country_term_counts[country].update(
        relevant_tokens
    )

In [207]:
countries = sorted(
    country_total_tokens.keys()
)

terms_69 = sorted(
    retained_terms_69
)

country_term_matrix_69 = pd.DataFrame(
    0.0,
    index=countries,
    columns=terms_69
)

for country in countries:

    for term, count in (
        country_term_counts[country].items()
    ):

        country_term_matrix_69.loc[
            country,
            term
        ] = count


print(
    "Country × term matrix:",
    country_term_matrix_69.shape
)

Country × term matrix: (178, 69)


In [208]:
country_token_totals_series = pd.Series(
    {
        country: country_total_tokens[country]
        for country in countries
    }
)

country_relative_freq_69 = (
    country_term_matrix_69
    .div(
        country_token_totals_series,
        axis=0
    )
)

In [209]:
weight_map_69 = (
    semantic_purified_kgl
    .set_index("term")["weight"]
    .to_dict()
)

weights_69 = pd.Series(
    {
        term: weight_map_69[term]
        for term in terms_69
    }
)

weighted_country_matrix_69 = (
    country_relative_freq_69
    .mul(
        weights_69,
        axis=1
    )
)

In [210]:
total_weighted_score_69 = (
    weighted_country_matrix_69
    .sum(axis=1)
)

correlation_rows_69 = []

for term in terms_69:

    term_vector = (
        weighted_country_matrix_69[term]
    )

    rest_vector = (
        total_weighted_score_69
        - term_vector
    )

    if (
        term_vector.nunique() <= 1
        or rest_vector.nunique() <= 1
    ):
        r = np.nan

    else:
        r, _ = pearsonr(
            term_vector,
            rest_vector
        )

    original_row = (
        semantic_purified_kgl[
            semantic_purified_kgl["term"].eq(term)
        ]
        .iloc[0]
    )

    correlation_rows_69.append({
        "term": term,
        "is_seed": original_row["is_seed"],
        "sources": original_row["sources"],
        "weight": original_row["weight"],
        "token_count": original_row["token_count"],
        "internal_correlation_recomputed": r
    })


correlations_69 = pd.DataFrame(
    correlation_rows_69
)

In [211]:
print(
    correlations_69[
        "internal_correlation_recomputed"
    ]
    .describe(
        percentiles=[
            .10,
            .25,
            .50,
            .75,
            .90
        ]
    )
)

count    69.000000
mean      0.125612
std       0.134361
min      -0.074917
10%      -0.043678
25%       0.024744
50%       0.105464
75%       0.224018
90%       0.301358
max       0.465943
Name: internal_correlation_recomputed, dtype: float64


In [212]:
for threshold in [
    0.00,
    0.05,
    0.10,
    0.15,
    0.20
]:

    n = (
        correlations_69[
            "internal_correlation_recomputed"
        ]
        > threshold
    ).sum()

    print(
        f"r > {threshold:.2f}: {n} terms"
    )

r > 0.00: 59 terms
r > 0.05: 46 terms
r > 0.10: 35 terms
r > 0.15: 26 terms
r > 0.20: 18 terms


In [213]:
print(
    correlations_69
    .sort_values(
        "internal_correlation_recomputed",
        ascending=False
    )[
        [
            "term",
            "is_seed",
            "internal_correlation_recomputed",
            "token_count"
        ]
    ]
    .to_string(index=False)
)

          term  is_seed  internal_correlation_recomputed  token_count
       condemn    False                         0.465943          360
        reject     True                         0.444889          267
     reiterate    False                         0.400695          942
      denounce    False                         0.391224           58
      reaffirm     True                         0.370133          648
       insists    False                         0.361990           20
      resisted    False                         0.353705           20
     rejecting    False                         0.288271           54
        demand    False                         0.279454          501
      asserted    False                         0.274662           13
       rejects    False                         0.268406           60
        affirm    False                         0.256554          134
        resist     True                         0.235540           63
       request    Fa

In [214]:
CORRELATIONS_69_FILE = (
    PROJECT_ROOT
    / "data"
    / "purification"
    / "self_assertion_semantic69_recomputed_correlations.csv"
)

correlations_69.to_csv(
    CORRELATIONS_69_FILE,
    index=False
)

print(CORRELATIONS_69_FILE)

../data/purification/self_assertion_semantic69_recomputed_correlations.csv


In [215]:
correlations_69 = pd.read_csv(
    PROJECT_ROOT
    / "data"
    / "purification"
    / "self_assertion_semantic69_recomputed_correlations.csv"
)

FINAL_CORRELATION_THRESHOLD = 0.00

final_self_assertion_kgl = (
    correlations_69[
        (
            correlations_69[
                "internal_correlation_recomputed"
            ]
            > FINAL_CORRELATION_THRESHOLD
        )
        |
        (
            correlations_69["is_seed"]
        )
    ]
    .copy()
    .sort_values("term")
    .reset_index(drop=True)
)

print(
    "Final Self-Assertion KGL terms:",
    len(final_self_assertion_kgl)
)

print(
    "Retained theory seeds:",
    final_self_assertion_kgl["is_seed"].sum()
)

print(
    "\nSeeds retained despite r <= 0:"
)

print(
    final_self_assertion_kgl.loc[
        final_self_assertion_kgl["is_seed"]
        &
        (
            final_self_assertion_kgl[
                "internal_correlation_recomputed"
            ]
            <= 0
        ),
        [
            "term",
            "internal_correlation_recomputed"
        ]
    ].to_string(index=False)
)

Final Self-Assertion KGL terms: 62
Retained theory seeds: 13

Seeds retained despite r <= 0:
          term  internal_correlation_recomputed
        insist                        -0.004665
         ready                        -0.056412
self-assertion                        -0.069897


In [216]:
FINAL_KGL_FILE = (
    PROJECT_ROOT
    / "data"
    / "purification"
    / "self_assertion_kgl_FINAL.csv"
)

final_self_assertion_kgl.to_csv(
    FINAL_KGL_FILE,
    index=False
)

print("Saved final Self-Assertion KGL to:")
print(FINAL_KGL_FILE)

Saved final Self-Assertion KGL to:
../data/purification/self_assertion_kgl_FINAL.csv
